# Fine-Tuning VadCLIP Có Hướng Dẫn Bằng Description Trên Colab

Notebook này chỉ đóng vai trò runner để chạy các file Python trong `VadCLIP/src`. Source code model vẫn nằm trong thư mục `VadCLIP/src`, không được copy trực tiếp vào notebook.


## 1. Mount Google Drive Và Cấu Hình Đường Dẫn

Cell này mount Google Drive, khai báo thư mục dự án, thư mục feature, checkpoint baseline và các đường dẫn dùng chung cho những cell phía sau.


In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT
DESCRIPTION_JSON = PROJECT_ROOT / 'code' / 'ucf_gpt_video_descriptions.json'
USE_VADCLIP_CHECKPOINT = False
PRETRAINED_MODEL = PROJECT_ROOT / 'model_ucf.pth'
RESULTS_ROOT = PROJECT_ROOT / 'code' / 'results'
EVALUATE_RESULTS_DIR = RESULTS_ROOT / 'evaluate'
ANALYZE_RESULTS_DIR = RESULTS_ROOT / 'analyze'
for result_dir in [RESULTS_ROOT, EVALUATE_RESULTS_DIR, ANALYZE_RESULTS_DIR]:
    result_dir.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)

print('Project root:', PROJECT_ROOT)
print('Source dir:', SRC_DIR)
print('Drive feature root:', DRIVE_FEATURE_ROOT)
print('Feature root:', FEATURE_ROOT)
print('Use VadCLIP checkpoint:', USE_VADCLIP_CHECKPOINT)
print('Results root:', RESULTS_ROOT)
print('Evaluate results:', EVALUATE_RESULTS_DIR)
print('Analyze results:', ANALYZE_RESULTS_DIR)

## 2. Cài Đặt Dependencies

Cell này cài các thư viện cần thiết cho VadCLIP trên Colab, bao gồm CLIP tokenizer, sklearn, scipy và matplotlib.


In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib


## 3. Kiểm Tra Các File Bắt Buộc

Cell này kiểm tra các file/folder quan trọng trước khi train, ví dụ source code, feature root, list train/test, ground truth và checkpoint baseline.


In [ ]:
required_paths = [
    SRC_DIR / 'model_description.py',
    SRC_DIR / 'ucf_train_description.py',
    SRC_DIR / 'ucf_train_class_semantic.py',
    SRC_DIR / 'ucf_option_class_semantic.py',
    SRC_DIR / 'ucf_train_class_prototype.py',
    SRC_DIR / 'ucf_option_class_prototype.py',
    SRC_DIR / 'model_prompt_replacement.py',
    SRC_DIR / 'ucf_train_prompt_replacement.py',
    SRC_DIR / 'ucf_option_prompt_replacement.py',
    SRC_DIR / 'model_multi_prompt.py',
    SRC_DIR / 'ucf_train_multi_prompt.py',
    SRC_DIR / 'ucf_option_multi_prompt.py',
    SRC_DIR / 'ucf_evaluate.py',
    SRC_DIR / 'ucf_analyze_checkpoints.py',
    SRC_DIR / 'ucf_analyze_manual_prompts.py',
    SRC_DIR / 'utils' / 'dataset_description.py',
    SRC_DIR / 'utils' / 'prompt_replacement.py',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
    PRETRAINED_MODEL,
    DESCRIPTION_JSON,
    PROJECT_ROOT / 'code' / 'ucf_class_description_prototypes_vadclip_train.json',
    PROJECT_ROOT / 'code' / 'ucf_compact_class_prototypes.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x_v2.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v3.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v4.json',
    PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts_auc8715.json',
]
if USE_VADCLIP_CHECKPOINT:
    required_paths.append(PRETRAINED_MODEL)

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if not DRIVE_FEATURE_ROOT.exists() and available_feature_archive is None:
    archive_names = ', '.join(path.name for path in DRIVE_FEATURE_ARCHIVES)
    raise FileNotFoundError(f'Missing UCFClipFeatures folder or one archive: {archive_names}')

print('All required files exist.')


## 4. Copy Feature Sang Runtime Local

Cell này copy `UCFClipFeatures` từ Google Drive sang `/content` để giảm I/O khi train. Nên chạy cell này nếu bạn muốn train nhanh hơn trên Colab.


In [ ]:
import shutil
import subprocess
import time

subprocess.run(['df', '-h', '/content'], check=False)
available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if available_feature_archive is not None:
    subprocess.run(['du', '-sh', str(available_feature_archive)], check=False)
else:
    subprocess.run(['du', '-sh', str(DRIVE_FEATURE_ROOT)], check=False)

start = time.time()
if available_feature_archive is not None:
    local_archive = Path('/content') / available_feature_archive.name
    if not local_archive.exists() or local_archive.stat().st_size != available_feature_archive.stat().st_size:
        print('Copying archive to local runtime:', available_feature_archive)
        shutil.copy2(available_feature_archive, local_archive)
    else:
        print('Local archive already exists with matching size:', local_archive)

    print('Extracting archive:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)

    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive must contain top-level folder UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run([
            'rsync', '-ah', '--info=progress2',
            f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'
        ], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'Prepared local features in {(time.time() - start) / 60:.1f} minutes')
print('Feature root:', FEATURE_ROOT)


## 5. Kiểm Tra Độ Phủ Feature

Cell này kiểm tra các file `.npy` được list train/test tham chiếu có tồn tại trong `FEATURE_ROOT` hay không. Nên chạy trước khi train để tránh lỗi thiếu feature giữa chừng.


In [ ]:
import csv
from collections import Counter

def check_feature_coverage(csv_path, feature_root, preview=30):
    rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))
    missing = []
    for row in rows:
        path = feature_root / row['path']
        if not path.exists():
            missing.append(row)

    print(f'{csv_path.name}: rows={len(rows)}, missing_files={len(missing)}')
    if missing:
        print('Missing by label:', dict(Counter(row['label'] for row in missing)))
        for row in missing[:preview]:
            print(f"  {row.get('video_id', '')},{row['label']},{row['path']}")
        if len(missing) > preview:
            print(f'  ... and {len(missing) - preview} more')
        raise FileNotFoundError(f'{csv_path.name} references missing feature files. Re-upload UCFClipFeatures or rebuild the list from available files.')

for csv_path in [
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
]:
    check_feature_coverage(csv_path, FEATURE_ROOT)


## 6. Smoke Test Dataset

Cell này thử load một vài sample từ dataset để kiểm tra shape feature, label, video id và đường dẫn feature có đúng không.


In [ ]:
from utils.dataset_description import UCFDescriptionDataset

label_map = {
    'Normal': 'normal', 'Abuse': 'abuse', 'Arrest': 'arrest', 'Arson': 'arson',
    'Assault': 'assault', 'Burglary': 'burglary', 'Explosion': 'explosion',
    'Fighting': 'fighting', 'RoadAccidents': 'roadAccidents', 'Robbery': 'robbery',
    'Shooting': 'shooting', 'Shoplifting': 'shoplifting', 'Stealing': 'stealing',
    'Vandalism': 'vandalism'
}

train_list = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv'
dataset = UCFDescriptionDataset(256, str(train_list), False, label_map, str(FEATURE_ROOT), str(DESCRIPTION_JSON), normal=False)
feature, label, length, video_id, description = dataset[0]
print('Dataset size:', len(dataset))
print('Feature shape:', feature.shape)
print('Label:', label)
print('Length:', length)
print('Video ID:', video_id)
print('Description:', description[:200])

## 7. Fine-Tune Description-Guided VadCLIP

Cell này là pipeline fine-tune cũ dùng video description. Hiện tại chủ yếu giữ lại để tham chiếu lịch sử thí nghiệm; các hướng prompt/prototype mới nằm ở các section phía dưới.


In [ ]:
import shlex
import subprocess

def run_command(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

train_description_cmd = [
    'python', 'ucf_train_description.py',
    '--feature-root', str(FEATURE_ROOT),
    '--description-json', str(DESCRIPTION_JSON),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-desc', '0.003',
    '--lambda-contrastive', '0.003',
    '--lambda-distill', '0.05',
    '--desc-loss-type', 'contrastive',
    '--desc-pooling', 'topk',
    '--top-k-ratio', '0.15',
    '--contrastive-temperature', '0.07',
    '--contrastive-samples', 'all',
    '--use-desc-projection', 'true',
    '--distill-baseline', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--trainable-scope', 'all',
    '--lr', '1e-5',
    '--max-epoch', '3',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--cache-description-embeddings', 'true',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_contrastive',
    '--output-model-path', 'model/model_ucf_description_contrastive.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_description_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_description_cmd))
subprocess.run(train_description_cmd, check=True)

## 8. Đánh Giá Baseline Và Model Fine-Tuned

Cell này chạy evaluator để so sánh checkpoint baseline với model fine-tuned theo các metric của VadCLIP: AUC, AP, Ano-AUC và mAP.


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_description_contrastive.pth" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 9. Phân Tích Checkpoint Và Vẽ Diagnostics

Cell này chạy script phân tích checkpoint theo epoch, lưu bảng metric, biểu đồ score distribution, per-class drift và timeline visualization.


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_contrastive" \
  --description-model-path "model/model_ucf_description_contrastive.pth" \
  --output-dir "{EVALUATE_RESULTS_DIR / 'ucf_checkpoint_diagnostics_contrastive'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 10. Fine-Tune Theo Class Semantic

Cell này chạy hướng class semantic regularization: dùng description để tạo soft target theo class, sau đó regularize phân phối class từ `logits2`. Hướng này được giữ lại để đối chiếu kết quả.


In [ ]:
train_class_semantic_cmd = [
    'python', 'ucf_train_class_semantic.py',
    '--feature-root', str(FEATURE_ROOT),
    '--description-json', str(DESCRIPTION_JSON),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-sem', '0.05',
    '--semantic-alpha', '0.2',
    '--semantic-temperature', '0.07',
    '--video-logit-pooling', 'topk',
    '--top-k-ratio', '0.15',
    '--lr', '1e-5',
    '--max-epoch', '3',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--cache-semantic-targets', 'true',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_class_semantic',
    '--output-model-path', 'model/model_ucf_class_semantic.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_class_semantic_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_class_semantic_cmd))
subprocess.run(train_class_semantic_cmd, check=True)


## 11. Đánh Giá Model Class Semantic

Cell này đánh giá model class semantic bằng evaluator chung, inference vẫn không dùng description.


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_class_semantic.pth" \
  --description-model-type baseline \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 12. Phân Tích Checkpoint Class Semantic

Cell này tạo diagnostics cho các checkpoint của hướng class semantic để xem metric thay đổi theo epoch và class nào bị lệch.


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_class_semantic" \
  --description-model-path "model/model_ucf_class_semantic.pth" \
  --finetuned-model-type baseline \
  --output-dir "{EVALUATE_RESULTS_DIR / 'ucf_checkpoint_diagnostics_class_semantic'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 13. Fine-Tune Class Prototype Từ Scratch

Cell này chạy hướng class prototype branch từ scratch. Model không load `model_ucf.pth` để khởi tạo training; checkpoint baseline chỉ dùng cho đánh giá/so sánh sau train.


In [ ]:
import shlex
import subprocess

train_class_prototype_cmd = [
    'python', 'ucf_train_class_prototype.py',
    '--feature-root', str(FEATURE_ROOT),
    '--prototype-json', str(PROJECT_ROOT / 'code' / 'ucf_class_description_prototypes_vadclip_train.json'),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-proto-mil', '1.0',
    '--lambda-proto', '0.05',
    '--prototype-temperature', '0.07',
    '--prototype-mode', 'centroid',
    '--lr', '2e-5',
    '--max-epoch', '10',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_class_prototype_scratch',
    '--output-model-path', 'model/model_ucf_class_prototype_scratch.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_class_prototype_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_class_prototype_cmd))
subprocess.run(train_class_prototype_cmd, check=True)


## 14. Đánh Giá Model Class Prototype

Cell này đánh giá model class prototype sau khi train, so sánh trực tiếp với baseline VadCLIP.


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_class_prototype_scratch.pth" \
  --description-model-type baseline \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 15. Phân Tích Checkpoint Class Prototype

Cell này phân tích các checkpoint của hướng class prototype, bao gồm AUC, AP, Ano-AUC, mAP và per-class drift.


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_class_prototype_scratch" \
  --description-model-path "model/model_ucf_class_prototype_scratch.pth" \
  --finetuned-model-type baseline \
  --output-dir "{EVALUATE_RESULTS_DIR / 'ucf_checkpoint_diagnostics_class_prototype_scratch'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 16. Ablation Compact Prototype

Section này thử các compact prototype ngắn từ `ucf_compact_class_prototypes.json`. Đây là nhóm thí nghiệm cũ để kiểm tra prototype ngắn có ít gây nhiễu hơn prototype dài hay không.


In [ ]:
import shlex
import subprocess

COMPACT_PROTOTYPE_JSON = PROJECT_ROOT / 'code' / 'ucf_compact_class_prototypes.json'
compact_variants = {
    'keyword': ['keyword_phrase'],
    'action': ['compact_action_phrase'],
    'sentence': ['short_visual_sentence'],
    'all': ['keyword_phrase', 'compact_action_phrase', 'short_visual_sentence'],
}

# Mặc định chỉ chạy một cấu hình để tránh vô tình train 4 cấu hình liên tiếp.
# Đổi thành list(compact_variants) nếu cần chạy lại toàn bộ ablation.
selected_compact_variants = ['all']
RUN_COMPACT_EVALUATION = True
RUN_COMPACT_ANALYSIS = False  # Analyzer đánh giá lại baseline và mọi checkpoint nên chạy rất lâu.

unknown_variants = set(selected_compact_variants) - set(compact_variants)
if unknown_variants:
    raise ValueError(f'Compact variants không hợp lệ: {sorted(unknown_variants)}')

for variant_name in selected_compact_variants:
    prototype_types = compact_variants[variant_name]
    print('\n' + '=' * 80)
    print('Compact prototype variant:', variant_name, prototype_types)

    epoch_dir = f'model/epoch_checkpoints_compact_{variant_name}_scratch'
    output_model = f'model/model_ucf_compact_{variant_name}_scratch.pth'
    checkpoint_path = f'model/checkpoint_compact_{variant_name}_scratch.pth'
    save_cur_path = f'model/model_cur_compact_{variant_name}_scratch.pth'
    diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_compact_{variant_name}_scratch'

    train_cmd = [
        'python', 'ucf_train_class_prototype.py',
        '--feature-root', str(FEATURE_ROOT),
        '--prototype-json', str(COMPACT_PROTOTYPE_JSON),
        '--prototype-schema', 'compact',
        '--compact-prototype-types', *prototype_types,
        '--use-pretrained-model', 'false',
        '--train-list', '../list/ucf_CLIP_rgb_description.csv',
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--lambda-proto-mil', '0.3',
        '--lambda-proto', '0.01',
        '--prototype-temperature', '0.07',
        '--prototype-mode', 'centroid',
        '--lr', '1e-5',
        '--max-epoch', '3',
        '--eval-steps', '0',
        '--num-workers', '2',
        '--pin-memory', 'true',
        '--use-amp', 'false',
        '--epoch-checkpoint-dir', epoch_dir,
        '--output-model-path', output_model,
        '--checkpoint-path', checkpoint_path,
        '--save-cur-path', save_cur_path,
    ]

    print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
    run_command(train_cmd)

    if RUN_COMPACT_EVALUATION:
        eval_cmd = [
            'python', 'ucf_evaluate.py',
            '--feature-root', str(FEATURE_ROOT),
            '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
            '--baseline-model-path', str(PRETRAINED_MODEL),
            '--description-model-path', output_model,
            '--description-model-type', 'baseline',
            '--gt-path', '../list/gt_ucf.npy',
            '--gt-segment-path', '../list/gt_segment_ucf.npy',
            '--gt-label-path', '../list/gt_label_ucf.npy',
        ]
        print('Evaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
        run_command(eval_cmd)

    if RUN_COMPACT_ANALYSIS:
        analyze_cmd = [
            'python', 'ucf_analyze_checkpoints.py',
            '--feature-root', str(FEATURE_ROOT),
            '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
            '--baseline-model-path', str(PRETRAINED_MODEL),
            '--epoch-checkpoint-dir', epoch_dir,
            '--description-model-path', output_model,
            '--finetuned-model-type', 'baseline',
            '--output-dir', str(diagnostics_dir),
            '--gt-path', '../list/gt_ucf.npy',
            '--gt-segment-path', '../list/gt_segment_ucf.npy',
            '--gt-label-path', '../list/gt_label_ucf.npy',
            '--timeline-count', '6',
        ]
        print('Analyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
        run_command(analyze_cmd)


## 17. Thí Nghiệm Prototype Prompt Replacement

Section này thử thay class prompt bằng prototype prompt. CLIP vẫn pretrained/frozen; phần VadCLIP-specific được train từ scratch trong các cấu hình hiện tại.


In [ ]:
import shlex
import subprocess

PROMPT_REPLACEMENT_PROTOTYPE_JSON = PROJECT_ROOT / 'code' / 'ucf_compact_class_prototypes.json'
prompt_replacement_variants = {
    'prototype_only': 'prototype_only',
    'class_plus': 'class_plus_prototype',
}

for variant_name, replacement_mode in prompt_replacement_variants.items():
    print('\n' + '=' * 80)
    print('Prompt replacement variant:', variant_name, replacement_mode)

    epoch_dir = f'model/epoch_checkpoints_prompt_replace_{variant_name}_scratch'
    output_model = f'model/model_ucf_prompt_replace_{variant_name}_scratch.pth'
    checkpoint_path = f'model/checkpoint_prompt_replace_{variant_name}_scratch.pth'
    save_cur_path = f'model/model_cur_prompt_replace_{variant_name}_scratch.pth'
    diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_prompt_replace_{variant_name}_scratch'

    train_cmd = [
        'python', 'ucf_train_prompt_replacement.py',
        '--feature-root', str(FEATURE_ROOT),
        '--prototype-json', str(PROMPT_REPLACEMENT_PROTOTYPE_JSON),
        '--prompt-replacement-mode', replacement_mode,
        '--compact-prototype-types', 'keyword_phrase', 'compact_action_phrase', 'short_visual_sentence',
        '--use-pretrained-model', 'false',
        '--train-list', '../list/ucf_CLIP_rgb_description.csv',
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--lr', '2e-5',
        '--max-epoch', '10',
        '--scheduler-milestones', '4', '8',
        '--eval-steps', '0',
        '--num-workers', '4',
        '--pin-memory', 'true',
        '--use-amp', 'false',
        '--epoch-checkpoint-dir', epoch_dir,
        '--output-model-path', output_model,
        '--checkpoint-path', checkpoint_path,
        '--save-cur-path', save_cur_path,
    ]
    print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
    subprocess.run(train_cmd, check=True)

    eval_cmd = [
        'python', 'ucf_evaluate.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--description-model-path', output_model,
        '--description-model-type', 'prompt_replacement',
        '--prompt-replacement-mode', replacement_mode,
        '--prototype-json', str(PROMPT_REPLACEMENT_PROTOTYPE_JSON),
        '--compact-prototype-types', 'keyword_phrase', 'compact_action_phrase', 'short_visual_sentence',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
    ]
    print('Evaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
    subprocess.run(eval_cmd, check=True)

    analyze_cmd = [
        'python', 'ucf_analyze_checkpoints.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--epoch-checkpoint-dir', epoch_dir,
        '--description-model-path', output_model,
        '--finetuned-model-type', 'prompt_replacement',
        '--prompt-replacement-mode', replacement_mode,
        '--prototype-json', str(PROMPT_REPLACEMENT_PROTOTYPE_JSON),
        '--compact-prototype-types', 'keyword_phrase', 'compact_action_phrase', 'short_visual_sentence',
        '--output-dir', str(diagnostics_dir),
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--timeline-count', '6',
    ]
    print('Analyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
    subprocess.run(analyze_cmd, check=True)


## 18. Multi-Prompt Training Từ Một File Prompt

Cell này train model bằng một file prompt cụ thể ở chế độ `prototype_only`, tức là prompt được đưa trực tiếp vào text encoder và không tự thêm tên class phía trước. Mỗi class có nhiều prompt, model tạo alignment map `[B,T,14,K]`, sau đó aggregate thành `logits2 [B,T,14]`.

Muốn đổi file prompt thì sửa `multi_prompt_variants` và `selected_multi_prompt_variants` trong cell bên dưới. Các regularizer chống tăng score negative cũng đã được thêm vào cell này và có thể bật bằng cách đổi lambda.


In [ ]:
import shlex
import subprocess

def run_command(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

multi_prompt_variants = {
    'prototype_only_10x': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '10',
        'scheduler_milestones': ['4', '8'],
    },
    'prototype_only_10x_v2': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x_v2.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '3',
        'scheduler_milestones': ['2'],
    },
    'prototype_only_5x_v3': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v3.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '3',
        'scheduler_milestones': ['2'],
    },
    'prototype_only_5x_v4': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v4.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '3',
        'scheduler_milestones': ['2'],
    },
}

# Default: run only the latest 5-prompt prototype-only v4 ablation so old runs are not repeated accidentally.
# To rerun older variants, add their names here, e.g. ['prototype_only_5x_v3', 'prototype_only_5x_v4'].
selected_multi_prompt_variants = ['prototype_only_5x_v4']

for variant_name in selected_multi_prompt_variants:
    cfg = multi_prompt_variants[variant_name]
    print('\n' + '=' * 80)
    print('Multi-prompt variant:', variant_name)

    epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
    output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
    checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
    save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
    diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'

    common_prompt_args = [
        '--prototype-source', cfg['prototype_source'],
        '--prototype-json', str(cfg['prototype_json']),
        '--prompt-mode', cfg['prompt_mode'],
        *cfg['type_args'],
    ]

    train_cmd = [
        'python', 'ucf_train_multi_prompt.py',
        '--feature-root', str(FEATURE_ROOT),
        *common_prompt_args,
        '--use-pretrained-model', 'false',
        '--train-list', '../list/ucf_CLIP_rgb_description.csv',
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--lambda-consistency', '0.01',
        '--consistency-top-k-ratio', '0.15',
        '--regularize-score-branch', 'both',
        '--regularize-top-k-ratio', '0.15',
        '--lambda-normal-topk', '0.0',
        '--lambda-abnormal-bg', '0.0',
        '--lambda-ranking-margin', '0.0',
        '--ranking-margin', '0.20',
        '--lr', '2e-5',
        '--max-epoch', cfg['max_epoch'],
        '--scheduler-milestones', *cfg['scheduler_milestones'],
        '--eval-steps', '0',
        '--num-workers', '4',
        '--pin-memory', 'true',
        '--use-amp', 'false',
        '--epoch-checkpoint-dir', epoch_dir,
        '--output-model-path', output_model,
        '--checkpoint-path', checkpoint_path,
        '--save-cur-path', save_cur_path,
    ]
    print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
    run_command(train_cmd)

    eval_cmd = [
        'python', 'ucf_evaluate.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--description-model-path', output_model,
        '--description-model-type', 'multi_prompt',
        *common_prompt_args,
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
    ]
    print('Evaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
    run_command(eval_cmd)

    analyze_cmd = [
        'python', 'ucf_analyze_checkpoints.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--epoch-checkpoint-dir', epoch_dir,
        '--description-model-path', output_model,
        '--finetuned-model-type', 'multi_prompt',
        *common_prompt_args,
        '--output-dir', str(diagnostics_dir),
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--timeline-count', '6',
    ]
    print('Analyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
    run_command(analyze_cmd)


## 19. Train VadCLIP Gốc Từ Scratch Không Regularize

Cell này train lại **VadCLIP public gốc** từ scratch trên UCF-Crime bằng các file baseline biệt lập (`ucf_train_public.py`, `ucf_test_public.py`, `ucf_option_public.py`, `utils/dataset_public.py`). Các thay đổi phục vụ những thí nghiệm khác không tham gia vào lần chạy này.

Mục tiêu của cell này là tạo mốc baseline tự train để so sánh với các thí nghiệm thêm regularization phía sau. Về mặt text prompt, model vẫn dùng đúng class prompt gốc của VadCLIP thông qua `get_prompt_text(label_map)`.

Giữ đúng cấu hình public: learning rate `2e-5`, 10 epoch, batch size 64, seed 234, scheduler giảm 10 lần ở epoch 4 và 8. Model được evaluate sau mỗi 1.280 mẫu train và checkpoint có AUC1 tốt nhất được nạp lại sau mỗi epoch, đúng như mã công khai của tác giả.


In [ ]:
import csv
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()


def write_absolute_feature_list(relative_csv, feature_root, output_csv, drop_missing=False):
    relative_csv = Path(relative_csv)
    feature_root = Path(feature_root)
    output_csv = Path(output_csv)
    rows = []
    missing = []
    with relative_csv.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for row in reader:
            item = dict(row)
            feature_path = (feature_root / item['path']).resolve()
            if not feature_path.exists():
                missing.append(str(feature_path))
                if drop_missing:
                    continue
            item['path'] = str(feature_path)
            rows.append(item)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    if missing:
        missing_report = output_csv.with_name(output_csv.stem + '_missing_features.txt')
        missing_report.write_text('\n'.join(missing), encoding='utf-8')
        print(f'Missing {len(missing)} features for {relative_csv.name}; report: {missing_report}')
        print('First missing examples:')
        for path in missing[:10]:
            print('  ', path)
        if not drop_missing:
            raise FileNotFoundError(
                f'{relative_csv.name} has {len(missing)} missing feature files. '
                f'Fix FEATURE_ROOT/copy features before evaluation. See {missing_report}'
            )
    with output_csv.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Wrote {len(rows)} rows to {output_csv}')


for script_name in [
    'ucf_train_public.py',
    'ucf_test_public.py',
    'ucf_option_public.py',
    'utils/dataset_public.py',
    'model.py',
]:
    script_path = SRC_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f'Missing original VadCLIP script: {script_path}')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
BASELINE_LIST_DIR = PROJECT_ROOT / 'code' / 'lists_baseline_colab'
train_abs_list = BASELINE_LIST_DIR / 'ucf_CLIP_rgb_absolute.csv'
test_abs_list = BASELINE_LIST_DIR / 'ucf_CLIP_rgbtest_absolute.csv'

baseline_feature_root = Path(LOCAL_FEATURE_ROOT) if 'LOCAL_FEATURE_ROOT' in globals() and Path(LOCAL_FEATURE_ROOT).exists() else Path(FEATURE_ROOT)
print('Baseline train feature root:', baseline_feature_root)
if baseline_feature_root != Path('/content/UCFClipFeatures'):
    print('Warning: baseline training is not using /content/UCFClipFeatures. If you extracted the zip, rerun the extract cell before this cell.')

for src_name, dst_path, drop_missing in [
    ('ucf_CLIP_rgb_relative.csv', train_abs_list, False),
    ('ucf_CLIP_rgbtest_relative.csv', test_abs_list, False),
]:
    src = LIST_DIR / src_name
    if not src.exists():
        raise FileNotFoundError(f'Missing relative list: {src}')
    write_absolute_feature_list(src, baseline_feature_root, dst_path, drop_missing=drop_missing)

Path('model').mkdir(exist_ok=True)

PAPER_BASELINE_MODEL = 'model/model_ucf_paper_scratch.pth'
PAPER_BASELINE_CHECKPOINT = 'model/checkpoint_paper_scratch.pth'

# Bắt đầu một lượt public baseline mới, không dùng checkpoint từ lần chạy trước.
stale_baseline_checkpoint = Path(PAPER_BASELINE_CHECKPOINT)
if stale_baseline_checkpoint.exists():
    stale_baseline_checkpoint.unlink()
    print('Removed stale baseline checkpoint:', stale_baseline_checkpoint)

train_argv = [
    'ucf_train_public.py',
    '--train-list', str(train_abs_list),
    '--test-list', str(test_abs_list),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--model-path', PAPER_BASELINE_MODEL,
    '--checkpoint-path', PAPER_BASELINE_CHECKPOINT,
]

launcher_path = Path('_run_ucf_train_paper_no_regularize.py')
launcher_path.write_text(
    'import runpy\n'
    'import sys\n'
    'from pathlib import Path\n'
    '\n'
    'src_dir = str(Path(__file__).resolve().parent)\n'
    'if src_dir not in sys.path:\n'
    '    sys.path.insert(0, src_dir)\n'
    '\n'
    'import torch\n'
    '_orig_torch_load = torch.load\n'
    '\n'
    'def _torch_load_compat(*args, **kwargs):\n'
    '    kwargs.setdefault("weights_only", False)\n'
    '    try:\n'
    '        return _orig_torch_load(*args, **kwargs)\n'
    '    except TypeError:\n'
    '        kwargs.pop("weights_only", None)\n'
    '        return _orig_torch_load(*args, **kwargs)\n'
    '\n'
    'torch.load = _torch_load_compat\n'
    '\n'
    'sys.argv = ' + repr(train_argv) + '\n'
    'runpy.run_path(str(Path(src_dir) / "ucf_train_public.py"), run_name="__main__")\n',
    encoding='utf-8',
)
train_baseline_cmd = ['python', str(launcher_path)]

print('Training original VadCLIP from scratch without extra regularization')
print('Effective argv:', ' '.join(shlex.quote(part) for part in train_argv))
print('Running:', ' '.join(shlex.quote(part) for part in train_baseline_cmd))
run_command(train_baseline_cmd)

test_baseline_cmd = [
    'python', 'ucf_test_public.py',
    '--model-path', PAPER_BASELINE_MODEL,
    '--test-list', str(test_abs_list),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
print('\nEvaluating saved paper-flow baseline model')
print('Running:', ' '.join(shlex.quote(part) for part in test_baseline_cmd))
run_command(test_baseline_cmd)

print('\nPaper-flow baseline outputs:')
print('  model:', PAPER_BASELINE_MODEL)
print('  checkpoint:', PAPER_BASELINE_CHECKPOINT)


## 20. Train VadCLIP Gốc Từ Scratch Với Normal Regularize

Cell này train lại **VadCLIP gốc** (`ucf_train.py` + `model.py`) từ scratch trên UCF-Crime, giữ nguyên flow của paper và chỉ thêm **normal top-k suppression** vào loss training. Mục tiêu là kiểm tra liệu ép score bất thường trên video Normal thấp xuống có giúp tăng classifier AUC hay không.

Về mặt text prompt, model vẫn dùng đúng class prompt gốc của VadCLIP thông qua `get_prompt_text(label_map)`. Không dùng keyword prompt, prototype prompt, multi-prompt wrapper hoặc prompt replacement trong cell này.

Cấu hình chính: train từ scratch, learning rate `1e-5`, 10 epoch, batch size 64, scheduler giảm 10× ở epoch 4 và 8. Normal top-k dùng warm-up 1 epoch, tức epoch đầu chưa bật regularizer. Cell này không evaluate liên tục theo step; thay vào đó model được evaluate một lần sau mỗi epoch để giảm thời gian chạy.


In [ ]:
import csv
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()


def write_absolute_feature_list(relative_csv, feature_root, output_csv, drop_missing=False):
    relative_csv = Path(relative_csv)
    feature_root = Path(feature_root)
    output_csv = Path(output_csv)
    rows = []
    missing = []
    with relative_csv.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for row in reader:
            item = dict(row)
            feature_path = (feature_root / item['path']).resolve()
            if not feature_path.exists():
                missing.append(str(feature_path))
                if drop_missing:
                    continue
            item['path'] = str(feature_path)
            rows.append(item)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    if missing:
        missing_report = output_csv.with_name(output_csv.stem + '_missing_features.txt')
        missing_report.write_text('\n'.join(missing), encoding='utf-8')
        print(f'Missing {len(missing)} features for {relative_csv.name}; report: {missing_report}')
        print('First missing examples:')
        for path in missing[:10]:
            print('  ', path)
        if not drop_missing:
            raise FileNotFoundError(
                f'{relative_csv.name} has {len(missing)} missing feature files. '
                f'Fix FEATURE_ROOT/copy features before evaluation. See {missing_report}'
            )
    with output_csv.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Wrote {len(rows)} rows to {output_csv}')
    if drop_missing and missing:
        print(f'Dropped {len(missing)} missing training rows to avoid dataloader FileNotFoundError.')


for script_name in ['ucf_train.py', 'ucf_test.py', 'model.py']:
    script_path = SRC_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f'Missing original VadCLIP script: {script_path}')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
BASELINE_LIST_DIR = PROJECT_ROOT / 'code' / 'lists_baseline_colab'
train_abs_list = BASELINE_LIST_DIR / 'ucf_CLIP_rgb_absolute.csv'
test_abs_list = BASELINE_LIST_DIR / 'ucf_CLIP_rgbtest_absolute.csv'

baseline_feature_root = Path(LOCAL_FEATURE_ROOT) if 'LOCAL_FEATURE_ROOT' in globals() and Path(LOCAL_FEATURE_ROOT).exists() else Path(FEATURE_ROOT)
print('Baseline train feature root:', baseline_feature_root)
if baseline_feature_root != Path('/content/UCFClipFeatures'):
    print('Warning: baseline training is not using /content/UCFClipFeatures. If you extracted the zip, rerun the extract cell before this cell.')

for src_name, dst_path, drop_missing in [
    ('ucf_CLIP_rgb_relative.csv', train_abs_list, False),
    ('ucf_CLIP_rgbtest_relative.csv', test_abs_list, False),
]:
    src = LIST_DIR / src_name
    if not src.exists():
        raise FileNotFoundError(f'Missing relative list: {src}')
    write_absolute_feature_list(src, baseline_feature_root, dst_path, drop_missing=drop_missing)

Path('model').mkdir(exist_ok=True)

LAMBDA_NORMAL_TOPK = 0.05
NORMAL_TOPK_WARMUP_EPOCHS = 1
REGULARIZE_SCORE_BRANCH = 'both'
REGULARIZE_TOP_K_RATIO = 0.15

NORMAL_REG_MODEL = 'model/model_ucf_paper_normal_topk_scratch.pth'
NORMAL_REG_CHECKPOINT = 'model/checkpoint_paper_normal_topk_scratch.pth'
NORMAL_REG_TOPK_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'topk_tracking_normal_topk'
NORMAL_REG_EPOCH_DIR = 'model/epoch_checkpoints_paper_normal_topk_scratch'

# Note: original ucf_option.py keeps some numeric CLI args without type,
# so this launcher casts them without changing the paper training flow.
train_argv = [
    'ucf_train.py',
    '--train-list', str(train_abs_list),
    '--test-list', str(test_abs_list),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lr', '1e-5',
    '--max-epoch', '10',
    '--batch-size', '64',
    '--seed', '234',
    '--model-path', NORMAL_REG_MODEL,
    '--checkpoint-path', NORMAL_REG_CHECKPOINT,
    '--eval-steps', '0',
    '--eval-each-epoch', 'True',
    '--lambda-normal-topk', str(LAMBDA_NORMAL_TOPK),
    '--normal-topk-warmup-epochs', str(NORMAL_TOPK_WARMUP_EPOCHS),
    '--regularize-score-branch', REGULARIZE_SCORE_BRANCH,
    '--regularize-top-k-ratio', str(REGULARIZE_TOP_K_RATIO),
    '--track-topk',
    '--topk-tracking-output-dir', str(NORMAL_REG_TOPK_DIR),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--save-epoch-checkpoints',
    '--epoch-checkpoint-dir', NORMAL_REG_EPOCH_DIR,
]

launcher_path = Path('_run_ucf_train_paper_normal_topk.py')
launcher_path.write_text(
    'import runpy\n'
    'import sys\n'
    'from pathlib import Path\n'
    '\n'
    'src_dir = str(Path(__file__).resolve().parent)\n'
    'if src_dir not in sys.path:\n'
    '    sys.path.insert(0, src_dir)\n'
    '\n'
    'import torch\n'
    '_orig_torch_load = torch.load\n'
    '\n'
    'def _torch_load_compat(*args, **kwargs):\n'
    '    kwargs.setdefault("weights_only", False)\n'
    '    try:\n'
    '        return _orig_torch_load(*args, **kwargs)\n'
    '    except TypeError:\n'
    '        kwargs.pop("weights_only", None)\n'
    '        return _orig_torch_load(*args, **kwargs)\n'
    '\n'
    'torch.load = _torch_load_compat\n'
    '\n'
    'import ucf_option\n'
    '\n'
    'def _has_option(parser, option):\n'
    '    return any(option in action.option_strings for action in parser._actions)\n'
    '\n'
    'def _ensure_arg(option, *args, **kwargs):\n'
    '    if not _has_option(ucf_option.parser, option):\n'
    '        ucf_option.parser.add_argument(option, *args, **kwargs)\n'
    '\n'
    '_ensure_arg("--training-mode", default="original", choices=["original", "abnormal_topk_epoch_eval"])\n'
    '_ensure_arg("--eval-steps", default=1280, type=int)\n'
    '_ensure_arg("--eval-each-epoch", default=False, type=bool)\n'
    '_ensure_arg("--lambda-normal-topk", default=0.0, type=float)\n'
    '_ensure_arg("--normal-topk-warmup-epochs", default=0, type=int)\n'
    '_ensure_arg("--lambda-abnormal-bg", default=0.0, type=float)\n'
    '_ensure_arg("--abnormal-bg-margin", default=0.20, type=float)\n'
    '_ensure_arg("--abnormal-bg-warmup-epochs", default=0, type=int)\n'
    '_ensure_arg("--regularize-score-branch", default="both", choices=["classifier", "alignment", "both"])\n'
    '_ensure_arg("--regularize-top-k-ratio", default=0.15, type=float)\n'
    '_ensure_arg("--save-epoch-checkpoints", action="store_true")\n'
    '_ensure_arg("--epoch-checkpoint-dir", default="model/epoch_checkpoints")\n'
    'sys.argv = ' + repr(train_argv) + '\n'
    '_orig_parse_args = ucf_option.parser.parse_args\n'
    '\n'
    'def _parse_args(*args, **kwargs):\n'
    '    parsed = _orig_parse_args(*args, **kwargs)\n'
    '    parsed.lr = float(parsed.lr)\n'
    '    parsed.scheduler_rate = float(parsed.scheduler_rate)\n'
    '    return parsed\n'
    '\n'
    'ucf_option.parser.parse_args = _parse_args\n'
    'runpy.run_path(str(Path(src_dir) / "ucf_train.py"), run_name="__main__")\n',
    encoding='utf-8',
)
train_baseline_cmd = ['python', str(launcher_path)]

print('Training original VadCLIP from scratch with normal top-k regularization')
print('Lambda normal top-k:', LAMBDA_NORMAL_TOPK)
print('Normal top-k warmup epochs:', NORMAL_TOPK_WARMUP_EPOCHS)
print('Regularize score branch:', REGULARIZE_SCORE_BRANCH)
print('Regularize top-k ratio:', REGULARIZE_TOP_K_RATIO)
print('Effective argv:', ' '.join(shlex.quote(part) for part in train_argv))
print('Running:', ' '.join(shlex.quote(part) for part in train_baseline_cmd))
run_command(train_baseline_cmd)

test_baseline_cmd = [
    'python', 'ucf_test.py',
    '--model-path', NORMAL_REG_MODEL,
    '--test-list', str(test_abs_list),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
print('\nEvaluating saved model with ucf_test.py')
print('Running:', ' '.join(shlex.quote(part) for part in test_baseline_cmd))
run_command(test_baseline_cmd)

print('\nNormal-topk paper-flow outputs:')
print('  model:', NORMAL_REG_MODEL)
print('  checkpoint:', NORMAL_REG_CHECKPOINT)
print('  top-k tracking:', NORMAL_REG_TOPK_DIR)
print('  epoch checkpoints:', SRC_DIR / NORMAL_REG_EPOCH_DIR)


## 21. Train VadCLIP Gốc Từ Scratch Với Abnormal Top-K, Evaluate Sau Mỗi Epoch

Cell này train lại **VadCLIP gốc** từ scratch, không dùng prompt/prototype, và chỉ thêm **abnormal top-k/background suppression** vào loss. Khác với flow gốc, cell này không evaluate liên tục trong epoch; model chỉ được evaluate một lần sau mỗi epoch để giảm thời gian chạy.

Regularizer chỉ áp dụng trên video Abnormal: chọn nhóm snippet có anomaly score cao nhất và nhóm snippet có anomaly score thấp nhất, sau đó ép hai nhóm tách nhau bằng margin. Normal top-k không được bật trong cell này.


In [ ]:
import csv
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()


def write_absolute_feature_list_if_needed(relative_csv, feature_root, output_csv):
    relative_csv = Path(relative_csv)
    feature_root = Path(feature_root)
    output_csv = Path(output_csv)
    rows = []
    missing = []
    with relative_csv.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for row in reader:
            item = dict(row)
            feature_path = (feature_root / item['path']).resolve()
            if not feature_path.exists():
                missing.append(str(feature_path))
            item['path'] = str(feature_path)
            rows.append(item)
    if missing:
        missing_report = output_csv.with_name(output_csv.stem + '_missing_features.txt')
        missing_report.write_text('\n'.join(missing), encoding='utf-8')
        print(f'Missing {len(missing)} features for {relative_csv.name}; report: {missing_report}')
        print('First missing examples:')
        for missing_path in missing[:10]:
            print('  ', missing_path)
        raise FileNotFoundError(f'{relative_csv.name} has {len(missing)} missing feature files.')
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    with output_csv.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Wrote {len(rows)} rows to {output_csv}')


for script_name in ['ucf_train.py', 'ucf_test.py', 'model.py']:
    script_path = SRC_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f'Missing original VadCLIP script: {script_path}')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
BASELINE_LIST_DIR = PROJECT_ROOT / 'code' / 'lists_baseline_colab'
train_abs_list = BASELINE_LIST_DIR / 'ucf_CLIP_rgb_absolute.csv'
test_abs_list = BASELINE_LIST_DIR / 'ucf_CLIP_rgbtest_absolute.csv'

baseline_feature_root = Path(LOCAL_FEATURE_ROOT) if 'LOCAL_FEATURE_ROOT' in globals() and Path(LOCAL_FEATURE_ROOT).exists() else Path(FEATURE_ROOT)
print('Abnormal-topk epoch-eval feature root:', baseline_feature_root)
if baseline_feature_root != Path('/content/UCFClipFeatures'):
    print('Warning: training is not using /content/UCFClipFeatures. If you extracted the zip, rerun the extract cell before this cell.')

write_absolute_feature_list_if_needed(LIST_DIR / 'ucf_CLIP_rgb_relative.csv', baseline_feature_root, train_abs_list)
write_absolute_feature_list_if_needed(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', baseline_feature_root, test_abs_list)

Path('model').mkdir(exist_ok=True)

LAMBDA_ABNORMAL_BG = 0.05
ABNORMAL_BG_MARGIN = 0.15
ABNORMAL_BG_WARMUP_EPOCHS = 1
REGULARIZE_SCORE_BRANCH = 'both'  # classifier | alignment | both
REGULARIZE_TOP_K_RATIO = 0.15

ABNORMAL_EPOCH_EVAL_MODEL = 'model/model_ucf_paper_abnormal_topk_epoch_eval_scratch.pth'
ABNORMAL_EPOCH_EVAL_CHECKPOINT = 'model/checkpoint_paper_abnormal_topk_epoch_eval_scratch.pth'
ABNORMAL_TOPK_TRACKING_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'topk_tracking_abnormal_topk'

train_argv = [
    'ucf_train.py',
    '--training-mode', 'abnormal_topk_epoch_eval',
    '--train-list', str(train_abs_list),
    '--test-list', str(test_abs_list),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lr', '1e-5',
    '--max-epoch', '10',
    '--batch-size', '64',
    '--seed', '234',
    '--model-path', ABNORMAL_EPOCH_EVAL_MODEL,
    '--checkpoint-path', ABNORMAL_EPOCH_EVAL_CHECKPOINT,
    '--eval-steps', '0',
    '--lambda-normal-topk', '0.0',
    '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
    '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
    '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
    '--regularize-score-branch', REGULARIZE_SCORE_BRANCH,
    '--regularize-top-k-ratio', str(REGULARIZE_TOP_K_RATIO),
    '--track-topk',
    '--topk-tracking-output-dir', str(ABNORMAL_TOPK_TRACKING_DIR),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
]

launcher_path = Path('_run_ucf_train_paper_abnormal_topk_epoch_eval.py')
launcher_path.write_text(
    'import runpy\n'
    'import sys\n'
    'from pathlib import Path\n'
    '\n'
    'src_dir = str(Path(__file__).resolve().parent)\n'
    'if src_dir not in sys.path:\n'
    '    sys.path.insert(0, src_dir)\n'
    '\n'
    'import torch\n'
    '_orig_torch_load = torch.load\n'
    '\n'
    'def _torch_load_compat(*args, **kwargs):\n'
    '    kwargs.setdefault("weights_only", False)\n'
    '    try:\n'
    '        return _orig_torch_load(*args, **kwargs)\n'
    '    except TypeError:\n'
    '        kwargs.pop("weights_only", None)\n'
    '        return _orig_torch_load(*args, **kwargs)\n'
    '\n'
    'torch.load = _torch_load_compat\n'
    '\n'
    'import ucf_option\n'
    '\n'
    'def _has_option(parser, option):\n'
    '    return any(option in action.option_strings for action in parser._actions)\n'
    '\n'
    'def _ensure_arg(option, *args, **kwargs):\n'
    '    if not _has_option(ucf_option.parser, option):\n'
    '        ucf_option.parser.add_argument(option, *args, **kwargs)\n'
    '\n'
    '_ensure_arg("--training-mode", default="original", choices=["original", "abnormal_topk_epoch_eval"])\n'
    '_ensure_arg("--eval-steps", default=1280, type=int)\n'
    '_ensure_arg("--eval-each-epoch", default=False, type=bool)\n'
    '_ensure_arg("--lambda-normal-topk", default=0.0, type=float)\n'
    '_ensure_arg("--normal-topk-warmup-epochs", default=0, type=int)\n'
    '_ensure_arg("--lambda-abnormal-bg", default=0.0, type=float)\n'
    '_ensure_arg("--abnormal-bg-margin", default=0.20, type=float)\n'
    '_ensure_arg("--abnormal-bg-warmup-epochs", default=0, type=int)\n'
    '_ensure_arg("--regularize-score-branch", default="both", choices=["classifier", "alignment", "both"])\n'
    '_ensure_arg("--regularize-top-k-ratio", default=0.15, type=float)\n'
    'sys.argv = ' + repr(train_argv) + '\n'
    '_orig_parse_args = ucf_option.parser.parse_args\n'
    '\n'
    'def _parse_args(*args, **kwargs):\n'
    '    parsed = _orig_parse_args(*args, **kwargs)\n'
    '    parsed.lr = float(parsed.lr)\n'
    '    parsed.scheduler_rate = float(parsed.scheduler_rate)\n'
    '    return parsed\n'
    '\n'
    'ucf_option.parser.parse_args = _parse_args\n'
    'runpy.run_path(str(Path(src_dir) / "ucf_train.py"), run_name="__main__")\n',
    encoding='utf-8',
)

train_cmd = ['python', str(launcher_path)]
print('Training original VadCLIP from scratch with abnormal top-k suppression and epoch-level evaluation')
print('Lambda abnormal bg:', LAMBDA_ABNORMAL_BG)
print('Abnormal bg margin:', ABNORMAL_BG_MARGIN)
print('Abnormal bg warmup epochs:', ABNORMAL_BG_WARMUP_EPOCHS)
print('Regularize score branch:', REGULARIZE_SCORE_BRANCH)
print('Regularize top-k ratio:', REGULARIZE_TOP_K_RATIO)
print('Effective argv:', ' '.join(shlex.quote(part) for part in train_argv))
print('Running:', ' '.join(shlex.quote(part) for part in train_cmd))
run_command(train_cmd)

print('Saved best epoch model to:', ABNORMAL_EPOCH_EVAL_MODEL)
print('Saved best checkpoint to:', ABNORMAL_EPOCH_EVAL_CHECKPOINT)
print('Top-k tracking:', ABNORMAL_TOPK_TRACKING_DIR)


## 21a. Train VadCLIP Gốc Với Ranking Margin

Train từ scratch với class name và learnable prompt gốc. Trong mỗi batch, Ranking Margin yêu cầu trung bình top 15% score của video abnormal cao hơn video Normal ít nhất 0.20. Loss áp dụng cho cả classifier và alignment, với trọng số 0.05 từ epoch 2. Kết quả chính là epoch 10; test AUC không được dùng để chọn checkpoint.

Chạy các cell mount Drive, cài thư viện và giải nén feature trước cell này. Cell tự tạo train/test list đầy đủ và lưu log cùng kết quả so sánh baseline tự train. Trong bảng evaluator, nhãn `Description-guided` là tên chung cho model thứ hai; ở đây nó chính là VadCLIP Ranking Margin.


In [ ]:
import csv
import subprocess
import sys
from pathlib import Path
import torch

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/content/drive/MyDrive/Finetune VadCLIP'))
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
RANK_FEATURE_ROOT = Path('/content/UCFClipFeatures')
RANK_TRAIN_DIR = PROJECT_ROOT / 'code' / 'results' / 'train' / 'ranking_margin'
RANK_EVAL_DIR = PROJECT_ROOT / 'code' / 'results' / 'evaluate' / 'ranking_margin'
RANK_MODEL_DIR = SRC_DIR / 'model'
RANK_MODEL_PATH = RANK_MODEL_DIR / 'model_ucf_ranking_margin_scratch.pth'
RANK_CHECKPOINT_PATH = RANK_MODEL_DIR / 'checkpoint_ranking_margin_scratch.pth'
RANK_EPOCH_DIR = RANK_MODEL_DIR / 'epoch_checkpoints_ranking_margin_scratch'
RANK_BASELINE_MODEL = RANK_MODEL_DIR / 'model_ucf_paper_scratch.pth'
LAMBDA_RANKING_MARGIN = 0.05
RANKING_MARGIN = 0.20
RANKING_WARMUP_EPOCHS = 1
RANK_MAX_EPOCH = 10

if not torch.cuda.is_available():
    raise RuntimeError('Cần bật GPU trên Colab để chạy VadCLIP.')
for required in [RANK_FEATURE_ROOT, RANK_BASELINE_MODEL, LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
                 LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', SRC_DIR / 'ucf_train.py',
                 SRC_DIR / 'ucf_option.py',
                 SRC_DIR / 'ucf_evaluate.py']:
    if not required.exists():
        raise FileNotFoundError(required)
if any('ranking_margin_epoch_eval' not in (SRC_DIR / name).read_text(encoding='utf-8')
       for name in ('ucf_train.py', 'ucf_option.py')):
    raise RuntimeError('Hãy upload phiên bản mới của ucf_train.py và ucf_option.py lên Drive.')
RANK_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
RANK_EVAL_DIR.mkdir(parents=True, exist_ok=True)
RANK_MODEL_DIR.mkdir(parents=True, exist_ok=True)

def rank_absolute_list(source, destination, expected_rows):
    with source.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        columns = reader.fieldnames
        rows = list(reader)
    if len(rows) != expected_rows:
        raise ValueError(f'{source.name}: cần {expected_rows} dòng, hiện có {len(rows)}.')
    for row in rows:
        feature_path = RANK_FEATURE_ROOT / row['path']
        if not feature_path.is_file():
            raise FileNotFoundError(feature_path)
        row['path'] = str(feature_path)
    with destination.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    print(f'{destination}: {len(rows)} video chunks')

RANK_TRAIN_LIST = RANK_TRAIN_DIR / 'train.csv'
RANK_TEST_LIST = RANK_TRAIN_DIR / 'test.csv'
rank_absolute_list(LIST_DIR / 'ucf_CLIP_rgb_relative.csv', RANK_TRAIN_LIST, 16100)
rank_absolute_list(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', RANK_TEST_LIST, 290)

def rank_run_and_log(command, output_path):
    print('Running:', ' '.join(str(part) for part in command), flush=True)
    with output_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Exit code {return_code}; xem log: {output_path}')

rank_train_command = [
    sys.executable, '-u', str(SRC_DIR / 'ucf_train.py'),
    '--training-mode', 'ranking_margin_epoch_eval',
    '--train-list', str(RANK_TRAIN_LIST), '--test-list', str(RANK_TEST_LIST),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
    '--model-path', str(RANK_MODEL_PATH), '--checkpoint-path', str(RANK_CHECKPOINT_PATH),
    '--save-cur-path', str(RANK_MODEL_DIR / 'model_cur_ranking_margin_scratch.pth'),
    '--epoch-checkpoint-dir', str(RANK_EPOCH_DIR),
    '--lr', '1e-5', '--max-epoch', str(RANK_MAX_EPOCH), '--batch-size', '64', '--seed', '234',
    '--lambda-ranking-margin', str(LAMBDA_RANKING_MARGIN),
    '--ranking-margin', str(RANKING_MARGIN),
    '--ranking-warmup-epochs', str(RANKING_WARMUP_EPOCHS),
    '--regularize-score-branch', 'both', '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', '0.0', '--lambda-abnormal-bg', '0.0',
]
rank_run_and_log(rank_train_command, RANK_TRAIN_DIR / 'train.log')

if not RANK_MODEL_PATH.is_file() or not (RANK_EPOCH_DIR / f'model_epoch_{RANK_MAX_EPOCH:03d}.pth').is_file():
    raise FileNotFoundError('Thiếu model hoặc checkpoint epoch cuối.')
rank_evaluate_command = [
    sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate.py'),
    '--feature-root', str(RANK_FEATURE_ROOT),
    '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
    '--baseline-model-path', str(RANK_BASELINE_MODEL),
    '--description-model-path', str(RANK_MODEL_PATH), '--description-model-type', 'baseline',
]
rank_run_and_log(rank_evaluate_command, RANK_EVAL_DIR / 'comparison_epoch_010.txt')
print('Model epoch 10:', RANK_MODEL_PATH)
print('Checkpoint theo epoch:', RANK_EPOCH_DIR)
print('Training log:', RANK_TRAIN_DIR / 'train.log')
print('Baseline so sánh epoch 10:', RANK_EVAL_DIR / 'comparison_epoch_010.txt')


## 21b. Ranking Margin mềm trên các cặp mẫu khó

Giữ VadCLIP gốc và ba loss gốc. Mỗi mẫu chọn top-k theo MIL gốc; từng mẫu abnormal được so với 25% mẫu normal điểm cao nhất trong batch. Loss softplus dùng margin 0,30 thay cho phép so trung bình hai nhóm. Learning rate giữ cố định ở 1e-5 suốt 10 epoch. Cell này độc lập với lượt Ranking Margin cũ và lưu output riêng.


In [ ]:
import csv
import subprocess
import sys
from pathlib import Path
import torch

ROOT = Path(globals().get('PROJECT_ROOT', '/content/drive/MyDrive/Finetune VadCLIP'))
SRC = ROOT / 'VadCLIP' / 'src'
LISTS = ROOT / 'VadCLIP' / 'list'
FEATURES = Path('/content/UCFClipFeatures')
TRAIN_OUT = ROOT / 'code' / 'results' / 'train' / 'ranking_hard_pair'
EVAL_OUT = ROOT / 'code' / 'results' / 'evaluate' / 'ranking_hard_pair'
MODEL_DIR = SRC / 'model'
MODEL = MODEL_DIR / 'model_ucf_ranking_hard_pair_scratch.pth'
EPOCH_DIR = MODEL_DIR / 'epoch_checkpoints_ranking_hard_pair_scratch'
BASELINE = MODEL_DIR / 'model_ucf_paper_scratch.pth'

if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU trong Colab trước khi train.')
for path in [FEATURES, BASELINE, SRC / 'ucf_train.py', SRC / 'ucf_option.py', SRC / 'ucf_evaluate.py']:
    if not path.exists():
        raise FileNotFoundError(path)
if any('ranking_hard_pair_epoch_eval' not in (SRC / name).read_text(encoding='utf-8')
       for name in ('ucf_train.py', 'ucf_option.py')):
    raise RuntimeError('Cần upload ucf_train.py và ucf_option.py mới lên Colab.')
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
EVAL_OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def absolute_feature_list(source, target, expected_rows):
    with source.open('r', encoding='utf-8', newline='') as stream:
        reader = csv.DictReader(stream)
        fields, rows = reader.fieldnames, list(reader)
    if len(rows) != expected_rows:
        raise ValueError(f'{source.name}: expected {expected_rows} rows, got {len(rows)}')
    for row in rows:
        path = FEATURES / row['path']
        if not path.is_file():
            raise FileNotFoundError(path)
        row['path'] = str(path)
    with target.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return target

def run_with_log(command, log_path):
    print('Running:', ' '.join(map(str, command)), flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=SRC, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f'Exit code {code}; xem log: {log_path}')

train_list = absolute_feature_list(LISTS / 'ucf_CLIP_rgb_relative.csv', TRAIN_OUT / 'train.csv', 16100)
test_list = absolute_feature_list(LISTS / 'ucf_CLIP_rgbtest_relative.csv', TRAIN_OUT / 'test.csv', 290)
train_cmd = [
    sys.executable, '-u', str(SRC / 'ucf_train.py'),
    '--training-mode', 'ranking_hard_pair_epoch_eval',
    '--train-list', str(train_list), '--test-list', str(test_list),
    '--gt-path', str(LISTS / 'gt_ucf.npy'),
    '--gt-segment-path', str(LISTS / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LISTS / 'gt_label_ucf.npy'),
    '--model-path', str(MODEL),
    '--checkpoint-path', str(MODEL_DIR / 'checkpoint_ranking_hard_pair_scratch.pth'),
    '--save-cur-path', str(MODEL_DIR / 'model_cur_ranking_hard_pair_scratch.pth'),
    '--epoch-checkpoint-dir', str(EPOCH_DIR),
    '--lr', '1e-5', '--scheduler-rate', '1.0', '--max-epoch', '10', '--batch-size', '64', '--seed', '234',
    '--lambda-ranking-margin', '0.05', '--ranking-margin', '0.30',
    '--ranking-warmup-epochs', '1', '--ranking-soft-temperature', '0.10',
    '--ranking-hard-normal-fraction', '0.25', '--regularize-score-branch', 'both',
]
run_with_log(train_cmd, TRAIN_OUT / 'train.log')
if not MODEL.is_file() or not (EPOCH_DIR / 'model_epoch_010.pth').is_file():
    raise FileNotFoundError('Thiếu model epoch 10 của lượt train mới.')
eval_cmd = [
    sys.executable, '-u', str(SRC / 'ucf_evaluate.py'),
    '--feature-root', str(FEATURES),
    '--test-list', str(LISTS / 'ucf_CLIP_rgbtest_relative.csv'),
    '--gt-path', str(LISTS / 'gt_ucf.npy'),
    '--gt-segment-path', str(LISTS / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LISTS / 'gt_label_ucf.npy'),
    '--baseline-model-path', str(BASELINE),
    '--description-model-path', str(MODEL), '--description-model-type', 'baseline',
]
run_with_log(eval_cmd, EVAL_OUT / 'comparison_epoch_010.txt')
print('Checkpoint epoch 10:', MODEL)
print('Training log:', TRAIN_OUT / 'train.log')
print('Evaluation:', EVAL_OUT / 'comparison_epoch_010.txt')


## 22. Training Với Prompt Bank Khôi Phục Từ Lần AUC 87.15

Cell này train lại từ scratch bằng prompt bank đã từng cho kết quả tốt nhất trước đó: `classifier_auc = 87.15` ở epoch 3.

Prompt bank này là weighted keyword 5x, được khôi phục vào file `ucf_weighted_class_prompts_auc8715.json`. Khi chạy mode `weighted`, model dùng toàn bộ 5 prompt/class và trọng số đi kèm. Đây là cấu hình nên chạy nếu muốn tái kiểm tra hoặc phát triển tiếp từ kết quả 87.15.

Cấu hình hiện tại:

- `PROMPT_TRAINING_MODE = 'weighted'`: dùng toàn bộ prompt và trọng số.
- `PROMPT_MODE = 'manual_caption'`: khi đưa vào model, prompt có dạng `class_name + prompt`.
- Train từ scratch, không load checkpoint fine-tune.
- Chạy tối đa 10 epoch, early stopping nếu AUC không cải thiện trong 3 epoch.
- Có bật `abnormal background separation` nhẹ để ép các đoạn score cao trong video bất thường tách khỏi đoạn nền score thấp. Chưa bật normal top-k suppression hoặc ranking margin.




In [ ]:
import json
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

# Modes:
# - weighted: use all prompts and prompt_weights from analyze output.
# - uniform: use all prompts from analyze output but ignore weights.
# - top1: keep only the highest-weight prompt per class for ablation.
PROMPT_TRAINING_MODE = 'weighted'  # weighted | uniform | top1
PROMPT_MODE = 'manual_caption'     # class_name + prompt text

# Abnormal background separation regularization.
# Loss này chỉ áp dụng cho video bất thường, ép vùng score cao tách khỏi vùng score thấp trong cùng video.
LAMBDA_ABNORMAL_BG = 0.05
ABNORMAL_BG_MARGIN = 0.15
ABNORMAL_BG_WARMUP_EPOCHS = 1

WEIGHTED_PROMPT_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts_auc8715.json'
if not WEIGHTED_PROMPT_JSON.exists():
    raise FileNotFoundError(
        f'Missing weighted prompt JSON: {WEIGHTED_PROMPT_JSON}\n'
        'Run analyze_vadclip_prompt_text_space.ipynb first to create this file.'
    )

with WEIGHTED_PROMPT_JSON.open('r', encoding='utf-8') as f:
    weighted_payload = json.load(f)

metadata = weighted_payload.get('metadata', {})
source_prompt_stem = Path(metadata.get('source_prompt_json', 'analyze_prompt')).stem
score_weights = metadata.get('score_weights', {})
variant_base = 'auc8715_weighted_keyword5x'

print('Prompt training mode:', PROMPT_TRAINING_MODE)
print('Weighted prompt JSON:', WEIGHTED_PROMPT_JSON)
print('Source prompt JSON:', metadata.get('source_prompt_json'))
print('Experiment variant base:', variant_base)
print('Embedding space:', metadata.get('embedding_space'))
print('Weighting method:', metadata.get('weighting_method'))
print('Score weights:', score_weights)
print('Prompts per class:', metadata.get('prompts_per_class'))
print('Prompt weight temperature:', metadata.get('prompt_weight_temperature'))
print('Lambda abnormal bg:', LAMBDA_ABNORMAL_BG)
print('Abnormal bg margin:', ABNORMAL_BG_MARGIN)
print('Abnormal bg warmup epochs:', ABNORMAL_BG_WARMUP_EPOCHS)

prompt_json_for_training = WEIGHTED_PROMPT_JSON
prompt_weights_json_for_training = None
variant_suffix = None

if PROMPT_TRAINING_MODE == 'weighted':
    prompt_weights_json_for_training = WEIGHTED_PROMPT_JSON
    variant_suffix = f'{variant_base}_weighted_abn_bg' if LAMBDA_ABNORMAL_BG > 0 else f'{variant_base}_weighted_no_regularize'
elif PROMPT_TRAINING_MODE == 'uniform':
    variant_suffix = f'{variant_base}_uniform_abn_bg' if LAMBDA_ABNORMAL_BG > 0 else f'{variant_base}_uniform_no_regularize'
elif PROMPT_TRAINING_MODE == 'top1':
    top1_payload = {
        'metadata': {
            **weighted_payload.get('metadata', {}),
            'name': 'top1_prompt_bank_from_weighted_analyze_output',
            'source_weighted_prompt_json': str(WEIGHTED_PROMPT_JSON),
            'training_mode': 'top1',
            'notes': [
                'This file is generated inside the Colab training notebook.',
                'For each class, it keeps only the prompt with the largest analyze weight.',
            ],
        },
        'classes': {},
    }
    print('\nTop-1 prompt selected from weights:')
    for class_name, prompts in weighted_payload['classes'].items():
        weights = weighted_payload['prompt_weights'][class_name]
        best_idx = max(range(len(weights)), key=lambda i: weights[i])
        top1_payload['classes'][class_name] = [prompts[best_idx]]
        print(f"{class_name:<13} weight={weights[best_idx]:.4f} #{best_idx}: {prompts[best_idx]}")
    prompt_json_for_training = PROJECT_ROOT / 'code' / 'ucf_top1_class_prompts_from_weighted.json'
    with prompt_json_for_training.open('w', encoding='utf-8') as f:
        json.dump(top1_payload, f, indent=2, ensure_ascii=False)
    variant_suffix = f'{variant_base}_top1_abn_bg' if LAMBDA_ABNORMAL_BG > 0 else f'{variant_base}_top1_no_regularize'
else:
    raise ValueError(f'Unknown PROMPT_TRAINING_MODE: {PROMPT_TRAINING_MODE}')

print('\nPrompt preview:')
for class_name, prompts in weighted_payload['classes'].items():
    weights = weighted_payload.get('prompt_weights', {}).get(class_name, [1.0 / len(prompts)] * len(prompts))
    pairs = sorted(zip(weights, prompts), reverse=True)
    preview = ' | '.join(f'{weight:.3f}: {prompt}' for weight, prompt in pairs[:3])
    print(f'{class_name:<13} {preview}')

variant_name = f'{PROMPT_MODE}_analyze_prompt_{variant_suffix}'
epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
diagnostics_dir = EVALUATE_RESULTS_DIR / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'
manual_prompt_diag_dir = ANALYZE_RESULTS_DIR / f'ucf_manual_prompt_diagnostics_{variant_name}_scratch'
prompt_topk_tracking_dir = ANALYZE_RESULTS_DIR / f'topk_tracking_{variant_name}_scratch'

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(prompt_json_for_training),
    '--prompt-mode', PROMPT_MODE,
]
if prompt_weights_json_for_training is not None:
    common_prompt_args.extend(['--prompt-weights-json', str(prompt_weights_json_for_training)])

train_cmd = [
    'python', 'ucf_train_multi_prompt.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', '0.01',
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', '0.0',
    '--lambda-abnormal-bg', str(LAMBDA_ABNORMAL_BG),
    '--abnormal-bg-margin', str(ABNORMAL_BG_MARGIN),
    '--abnormal-bg-warmup-epochs', str(ABNORMAL_BG_WARMUP_EPOCHS),
    '--lambda-ranking-margin', '0.0',
    '--ranking-margin', '0.20',
    '--lr', '2e-5',
    '--max-epoch', '10',
    '--scheduler-milestones', '4', '8',
    '--early-stopping-patience', '3',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--track-topk', 'true',
    '--topk-tracking-output-dir', str(prompt_topk_tracking_dir),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]
print('\n' + '=' * 80)
print('Analyze prompt training variant:', variant_name)
print('Prompt JSON:', prompt_json_for_training)
print('Prompt weights JSON:', prompt_weights_json_for_training)
print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
run_command(train_cmd)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
print('\nEvaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
run_command(eval_cmd)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
print('\nAnalyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
run_command(analyze_cmd)

manual_prompt_diag_cmd = [
    'python', 'ucf_analyze_manual_prompts.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--manual-model-path', output_model,
    '--prompt-json', str(prompt_json_for_training),
    '--prompt-mode', PROMPT_MODE,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--output-dir', str(manual_prompt_diag_dir),
]
if prompt_weights_json_for_training is not None:
    manual_prompt_diag_cmd.extend(['--prompt-weights-json', str(prompt_weights_json_for_training)])
print('\nManual prompt diagnostics:', ' '.join(shlex.quote(part) for part in manual_prompt_diag_cmd))
run_command(manual_prompt_diag_cmd)

print('\nAnalyze prompt training outputs:')
print('  mode:', PROMPT_TRAINING_MODE)
print('  model:', output_model)
print('  epoch checkpoints:', epoch_dir)
print('  checkpoint diagnostics:', diagnostics_dir)
print('  prompt diagnostics:', manual_prompt_diag_dir)
print('  top-k tracking:', prompt_topk_tracking_dir)




## 23. Phân Tích Manual Prompt

Cell này không train model. Nó phân tích manual CLIP-friendly prompts để tìm class, video hoặc prompt nào gây tụt metric. Mặc định phân tích checkpoint được khai báo trong `MANUAL_MODEL_TO_ANALYZE`; nếu muốn phân tích epoch khác, chỉ cần đổi path checkpoint tương ứng.


In [ ]:
import shlex
import subprocess

def run_command(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v4.json'
MANUAL_MODEL_TO_ANALYZE = 'model/epoch_checkpoints_multi_prompt_manual_caption_5x_v4_scratch/model_epoch_002_multi_prompt.pth'
MANUAL_PROMPT_DIAGNOSTICS_DIR = ANALYZE_RESULTS_DIR / 'ucf_manual_prompt_diagnostics_5x_v4'
# To analyze v3 epoch 2 instead:
# MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v3.json'
# MANUAL_MODEL_TO_ANALYZE = 'model/epoch_checkpoints_multi_prompt_manual_caption_5x_v3_scratch/model_epoch_002_multi_prompt.pth'
# MANUAL_PROMPT_DIAGNOSTICS_DIR = ANALYZE_RESULTS_DIR / 'ucf_manual_prompt_diagnostics_5x_v3'
# To analyze v2 epoch 2 instead:
# MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x_v2.json'
# MANUAL_MODEL_TO_ANALYZE = 'model/epoch_checkpoints_multi_prompt_manual_caption_10x_v2_scratch/model_epoch_002_multi_prompt.pth'
# MANUAL_PROMPT_DIAGNOSTICS_DIR = ANALYZE_RESULTS_DIR / 'ucf_manual_prompt_diagnostics_v2'
# To analyze v1 final instead:
# MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x.json'
# MANUAL_MODEL_TO_ANALYZE = 'model/model_ucf_multi_prompt_manual_caption_10x_scratch.pth'
# MANUAL_PROMPT_DIAGNOSTICS_DIR = ANALYZE_RESULTS_DIR / 'ucf_manual_prompt_diagnostics'

manual_prompt_diag_cmd = [
    'python', 'ucf_analyze_manual_prompts.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--manual-model-path', MANUAL_MODEL_TO_ANALYZE,
    '--prompt-json', str(MANUAL_PROMPT_JSON_TO_ANALYZE),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--output-dir', str(MANUAL_PROMPT_DIAGNOSTICS_DIR),
]
print('Running:', ' '.join(shlex.quote(part) for part in manual_prompt_diag_cmd))
run_command(manual_prompt_diag_cmd)


## 24. Thống Kê Score Separation Cho Các Checkpoint Đã Train

Cell này không train lại model. Nó chỉ load các checkpoint đã có trong Drive và tính các thống kê giúp giải thích vì sao AUC/mAP thay đổi: top-k overlap với ground truth, phân phối quantile của positive/negative score, và correlation giữa `logits1` với `logits2`.

Để tái sử dụng cho thí nghiệm khác, chỉ cần sửa danh sách `MODELS_TO_ANALYZE`: đổi `name`, `model_type`, `model_path`, và nếu là multi-prompt thì truyền thêm `prototype_json`, `prompt_weights_json`, `prototype_source`, `prompt_mode`.


In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

# Cell 24: reusable score-separation diagnostics.
# Chỉ cần sửa MODELS_TO_ANALYZE để phân tích checkpoint khác, không cần train lại.
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')
if 'ANALYZE_RESULTS_DIR' not in globals():
    ANALYZE_RESULTS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze'

os.chdir(SRC_DIR)
print('Working directory:', Path.cwd())

SCORE_SEPARATION_OUTPUT_DIR = ANALYZE_RESULTS_DIR / 'score_separation'
SCORE_SEPARATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTED_PROMPT_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts_auc8715.json'

MODELS_TO_ANALYZE = [
    {
        'name': 'baseline_self_train',
        'model_type': 'baseline',
        'model_path': str(SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'),
    },
    {
        'name': 'baseline_abnormal_topk',
        'model_type': 'baseline',
        'model_path': str(SRC_DIR / 'model' / 'model_ucf_paper_abnormal_topk_epoch_eval_scratch.pth'),
    },
    {
        'name': 'weighted_prompt_no_regularize',
        'model_type': 'multi_prompt',
        'model_path': str(
            SRC_DIR / 'model' /
            'model_ucf_multi_prompt_manual_caption_analyze_prompt_auc8715_weighted_keyword5x_weighted_no_regularize_scratch.pth'
        ),
        'prototype_source': 'manual',
        'prototype_json': str(WEIGHTED_PROMPT_JSON),
        'prompt_weights_json': str(WEIGHTED_PROMPT_JSON),
        'prompt_mode': 'manual_caption',
    },
    {
        'name': 'weighted_prompt_abnormal_topk',
        'model_type': 'multi_prompt',
        'model_path': str(
            SRC_DIR / 'model' /
            'model_ucf_multi_prompt_manual_caption_analyze_prompt_auc8715_weighted_keyword5x_weighted_abn_bg_scratch.pth'
        ),
        'prototype_source': 'manual',
        'prototype_json': str(WEIGHTED_PROMPT_JSON),
        'prompt_weights_json': str(WEIGHTED_PROMPT_JSON),
        'prompt_mode': 'manual_caption',
    },
]

available_models = []
print('Checking model paths for score-separation diagnostics:')
for config in MODELS_TO_ANALYZE:
    model_path = Path(config['model_path'])
    if model_path.exists():
        available_models.append(config)
        print(f"  FOUND   {config['name']}: {model_path}")
    else:
        print(f"  MISSING {config['name']}: {model_path}")

if not available_models:
    raise FileNotFoundError(
        'No checkpoint in MODELS_TO_ANALYZE exists. Update model_path values before running this cell.'
    )

models_config_path = SCORE_SEPARATION_OUTPUT_DIR / 'models_to_analyze.json'
with models_config_path.open('w', encoding='utf-8') as f:
    json.dump(available_models, f, indent=2, ensure_ascii=False)

score_separation_cmd = [
    'python', 'ucf_analyze_score_separation.py',
    '--models-json', str(models_config_path),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--top-k-ratio', '0.15',
    '--output-dir', str(SCORE_SEPARATION_OUTPUT_DIR),
]

print('\nRunning:', ' '.join(shlex.quote(part) for part in score_separation_cmd))
run_command(score_separation_cmd)

print('\nScore-separation output dir:', SCORE_SEPARATION_OUTPUT_DIR)
print('Main files:')
for filename in [
    'summary_by_model.csv',
    'score_quantiles_by_model.csv',
    'logits1_logits2_correlation.csv',
    'per_video_topk_overlap.csv',
    'per_class_score_separation.csv',
]:
    print(' ', SCORE_SEPARATION_OUTPUT_DIR / filename)


## 25. Phân Tích Topology Và Lỗi Xếp Hạng Ở Mức Snippet

Cell này phân tích trực tiếp các snippet thay vì lặp score ra 16 frame trước khi chọn top-k. Kết quả gồm index của snippet score cao, mức liên tục theo thời gian, bốn nhóm đúng/sai, độ đồng thuận giữa hai nhánh và số cặp positive-negative bị xếp ngược.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

# Cell 25: chỉ sửa danh sách model bên dưới khi muốn tái sử dụng cho checkpoint khác.
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')
if 'ANALYZE_RESULTS_DIR' not in globals():
    ANALYZE_RESULTS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze'

os.chdir(SRC_DIR)
print('Thư mục làm việc:', Path.cwd())

SNIPPET_TOPOLOGY_OUTPUT_DIR = ANALYZE_RESULTS_DIR / 'snippet_topology'
SNIPPET_TOPOLOGY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ưu tiên dùng đúng danh sách checkpoint của cell 24. Nếu chạy cell này độc lập,
# chương trình sẽ đọc cấu hình đã được cell 24 lưu lại.
if 'MODELS_TO_ANALYZE' in globals():
    snippet_models = MODELS_TO_ANALYZE
else:
    previous_config = ANALYZE_RESULTS_DIR / 'score_separation' / 'models_to_analyze.json'
    if not previous_config.exists():
        raise FileNotFoundError(
            'Không tìm thấy cấu hình model. Hãy chạy cell 24 trước hoặc khai báo MODELS_TO_ANALYZE.'
        )
    with previous_config.open('r', encoding='utf-8') as f:
        snippet_models = json.load(f)

available_models = []
print('Kiểm tra checkpoint cho phân tích snippet:')
for config in snippet_models:
    model_path = Path(config['model_path'])
    if model_path.exists():
        available_models.append(config)
        print(f"  FOUND   {config['name']}: {model_path}")
    else:
        print(f"  MISSING {config['name']}: {model_path}")

if not available_models:
    raise FileNotFoundError('Không có checkpoint hợp lệ để phân tích.')

snippet_models_path = SNIPPET_TOPOLOGY_OUTPUT_DIR / 'models_to_analyze.json'
with snippet_models_path.open('w', encoding='utf-8') as f:
    json.dump(available_models, f, indent=2, ensure_ascii=False)

snippet_topology_cmd = [
    'python', 'ucf_analyze_snippet_topology.py',
    '--models-json', str(snippet_models_path),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--top-k-ratios', '0.05,0.10,0.15',
    '--output-dir', str(SNIPPET_TOPOLOGY_OUTPUT_DIR),
]

print('\nĐang chạy:', ' '.join(shlex.quote(part) for part in snippet_topology_cmd))
run_command(snippet_topology_cmd)

print('\nKết quả phân tích snippet:', SNIPPET_TOPOLOGY_OUTPUT_DIR)
print('Các file chính:')
for filename in [
    'summary.csv',
    'snippet_index_scores.csv',
    'high_score_topology.csv',
    'high_score_index_sequences.json',
    'topk_four_groups.csv',
    'four_group_summary.csv',
    'branch_agreement.csv',
    'branch_agreement_summary.csv',
    'ground_truth_anomaly_topology.csv',
    'ground_truth_topology_by_class.csv',
    'per_class_snippet_summary.csv',
]:
    print(' ', SNIPPET_TOPOLOGY_OUTPUT_DIR / filename)


## 26. Phân Tích Visual Feature Và Class Attribution

Cell này so sánh true-high, false-high, missed-anomaly và true-low tại ba tầng visual của VadCLIP. Kết quả định lượng gồm centroid similarity, k-NN purity, linear probe chia fold theo video, similarity trong cùng video, feature drift và đóng góp của class/prompt.

In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

# Cell 26: phân tích feature của checkpoint có sẵn, không thực hiện training.
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')
if 'ANALYZE_RESULTS_DIR' not in globals():
    ANALYZE_RESULTS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze'

os.chdir(SRC_DIR)
print('Thư mục làm việc:', Path.cwd())

VISUAL_ANALYSIS_OUTPUT_DIR = ANALYZE_RESULTS_DIR / 'visual_feature_diagnostics'
VISUAL_ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kế thừa danh sách model từ cell 25 hoặc cell 24. Nếu chạy độc lập, đọc file cấu hình đã lưu.
if 'available_models' in globals() and available_models:
    visual_models = available_models
elif 'MODELS_TO_ANALYZE' in globals():
    visual_models = MODELS_TO_ANALYZE
else:
    previous_config = ANALYZE_RESULTS_DIR / 'snippet_topology' / 'models_to_analyze.json'
    if not previous_config.exists():
        previous_config = ANALYZE_RESULTS_DIR / 'score_separation' / 'models_to_analyze.json'
    if not previous_config.exists():
        raise FileNotFoundError(
            'Không tìm thấy cấu hình model. Hãy chạy cell 24/25 trước hoặc khai báo MODELS_TO_ANALYZE.'
        )
    with previous_config.open('r', encoding='utf-8') as f:
        visual_models = json.load(f)

valid_visual_models = []
print('Kiểm tra checkpoint cho phân tích visual feature:')
for config in visual_models:
    model_path = Path(config['model_path'])
    if model_path.exists():
        valid_visual_models.append(config)
        print(f"  FOUND   {config['name']}: {model_path}")
    else:
        print(f"  MISSING {config['name']}: {model_path}")

if not valid_visual_models:
    raise FileNotFoundError('Không có checkpoint hợp lệ để phân tích visual feature.')

visual_models_path = VISUAL_ANALYSIS_OUTPUT_DIR / 'models_to_analyze.json'
with visual_models_path.open('w', encoding='utf-8') as f:
    json.dump(valid_visual_models, f, indent=2, ensure_ascii=False)

visual_analysis_cmd = [
    'python', 'ucf_analyze_visual_features.py',
    '--models-json', str(visual_models_path),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--top-k-ratio', '0.15',
    '--max-samples-per-group', '1000',
    '--knn-neighbors', '5',
    '--output-dir', str(VISUAL_ANALYSIS_OUTPUT_DIR),
]

print('\nĐang chạy:', ' '.join(shlex.quote(part) for part in visual_analysis_cmd))
run_command(visual_analysis_cmd)

print('\nKết quả phân tích visual feature:', VISUAL_ANALYSIS_OUTPUT_DIR)
print('Các file chính:')
for filename in [
    'stage_centroid_similarity.csv',
    'stage_knn_purity.csv',
    'stage_linear_probe.csv',
    'within_video_group_similarity.csv',
    'stage_feature_drift.csv',
    'class_attribution_summary.csv',
    'class_confusion_by_group.csv',
    'snippet_class_attribution_<model>.csv',
    '<model>_feature_samples.npz',
    '<model>_stage_pca.png',
]:
    print(' ', VISUAL_ANALYSIS_OUTPUT_DIR / filename)


## 27. Xác Định Nguồn Lỗi: Classifier, Normal Bank Và Video Context

Cell này chỉ phân tích các checkpoint đã có, không training và không thay đổi loss. Ba kiểm tra gồm: final visual so với đầu vào classifier, normal visual memory bank xây từ train split Normal, và counterfactual video-context với nhiều giá trị gamma.

In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

# Cell 27: toàn bộ memory bank và gamma ablation chỉ dùng cho diagnostics.
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')
if 'ANALYZE_RESULTS_DIR' not in globals():
    ANALYZE_RESULTS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze'

os.chdir(SRC_DIR)
print('Thư mục làm việc:', Path.cwd())

FAILURE_ANALYSIS_OUTPUT_DIR = ANALYZE_RESULTS_DIR / 'failure_source_diagnostics'
FAILURE_ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kế thừa checkpoint từ các cell diagnostics trước. Có thể thay danh sách này để tái sử dụng.
if 'valid_visual_models' in globals() and valid_visual_models:
    failure_models = valid_visual_models
elif 'available_models' in globals() and available_models:
    failure_models = available_models
elif 'MODELS_TO_ANALYZE' in globals():
    failure_models = MODELS_TO_ANALYZE
else:
    previous_config = ANALYZE_RESULTS_DIR / 'visual_feature_diagnostics' / 'models_to_analyze.json'
    if not previous_config.exists():
        previous_config = ANALYZE_RESULTS_DIR / 'snippet_topology' / 'models_to_analyze.json'
    if not previous_config.exists():
        raise FileNotFoundError(
            'Không tìm thấy cấu hình model. Hãy chạy một cell diagnostics trước hoặc khai báo MODELS_TO_ANALYZE.'
        )
    with previous_config.open('r', encoding='utf-8') as f:
        failure_models = json.load(f)

valid_failure_models = []
print('Kiểm tra checkpoint cho failure-source diagnostics:')
for config in failure_models:
    model_path = Path(config['model_path'])
    if model_path.exists():
        valid_failure_models.append(config)
        print(f"  FOUND   {config['name']}: {model_path}")
    else:
        print(f"  MISSING {config['name']}: {model_path}")

if not valid_failure_models:
    raise FileNotFoundError('Không có checkpoint hợp lệ để phân tích.')

failure_models_path = FAILURE_ANALYSIS_OUTPUT_DIR / 'models_to_analyze.json'
with failure_models_path.open('w', encoding='utf-8') as f:
    json.dump(valid_failure_models, f, indent=2, ensure_ascii=False)

failure_analysis_cmd = [
    'python', 'ucf_analyze_failure_sources.py',
    '--models-json', str(failure_models_path),
    '--feature-root', str(FEATURE_ROOT),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--top-k-ratio', '0.15',
    '--context-gammas', '0,0.25,0.5,0.75,1',
    '--normal-prototypes', '32',
    '--max-normal-train-rows', '500',
    '--max-normal-bank-features', '50000',
    '--normal-batch-size', '8',
    '--max-samples-per-group', '1000',
    '--knn-neighbors', '5',
    '--output-dir', str(FAILURE_ANALYSIS_OUTPUT_DIR),
]

print('\nĐang chạy:', ' '.join(shlex.quote(part) for part in failure_analysis_cmd))
print('Lưu ý: cell này không training và không áp dụng normal bank vào model.')
run_command(failure_analysis_cmd)

print('\nKết quả failure-source diagnostics:', FAILURE_ANALYSIS_OUTPUT_DIR)
print('Các file chính:')
for filename in [
    'classifier_vs_feature.csv',
    'classifier_group_summary.csv',
    'normal_bank_group_similarity.csv',
    'normal_bank_separation_auc.csv',
    'normal_bank_per_class.csv',
    'normal_bank_metadata.csv',
    'normal_bank_sampled_rows.json',
    'normal_visual_bank_<model>.npz',
    'context_gamma_metrics.csv',
    'context_gamma_group_summary.csv',
    'context_gamma_per_class.csv',
    'context_gamma_sanity_check.csv',
]:
    print(' ', FAILURE_ANALYSIS_OUTPUT_DIR / filename)


## 28. Phân Tích Temporal Residual Từ CLIP Feature Có Sẵn

Cell này kiểm tra liệu mức thay đổi của CLIP feature theo thời gian có giúp phân biệt anomaly thật với false-high background hay không. Các descriptor gồm first difference, second difference, cosine change, local residual, local variance và change-point score. Toàn bộ phép tính dùng trực tiếp feature `.npy` hiện có, không đọc raw video và không chạy thêm video encoder. Ground truth chỉ dùng để thống kê sau inference, không dùng để thay đổi model.


In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')
if 'ANALYZE_RESULTS_DIR' not in globals():
    ANALYZE_RESULTS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze'

os.chdir(SRC_DIR)
analyzer_path = SRC_DIR / 'ucf_analyze_temporal_residuals.py'
if not analyzer_path.exists():
    raise FileNotFoundError(
        f'Không tìm thấy {analyzer_path}. Hãy upload file analyzer mới vào VadCLIP/src.'
    )
if not FEATURE_ROOT.exists():
    raise FileNotFoundError(
        f'Không tìm thấy feature root: {FEATURE_ROOT}. Hãy chạy cell giải nén feature trước.'
    )

TEMPORAL_RESIDUAL_OUTPUT_DIR = ANALYZE_RESULTS_DIR / 'temporal_residual_diagnostics'
TEMPORAL_RESIDUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kế thừa đúng các checkpoint đã dùng ở những phần diagnostics trước.
if 'valid_failure_models' in globals() and valid_failure_models:
    temporal_models = valid_failure_models
elif 'valid_visual_models' in globals() and valid_visual_models:
    temporal_models = valid_visual_models
elif 'available_models' in globals() and available_models:
    temporal_models = available_models
elif 'MODELS_TO_ANALYZE' in globals():
    temporal_models = MODELS_TO_ANALYZE
else:
    config_candidates = [
        ANALYZE_RESULTS_DIR / 'failure_source_diagnostics' / 'models_to_analyze.json',
        ANALYZE_RESULTS_DIR / 'visual_feature_diagnostics' / 'models_to_analyze.json',
        ANALYZE_RESULTS_DIR / 'snippet_topology' / 'models_to_analyze.json',
    ]
    config_path = next((path for path in config_candidates if path.exists()), None)
    if config_path is None:
        raise FileNotFoundError(
            'Không tìm thấy cấu hình model. Hãy chạy một cell diagnostics trước hoặc khai báo MODELS_TO_ANALYZE.'
        )
    with config_path.open('r', encoding='utf-8') as f:
        temporal_models = json.load(f)

valid_temporal_models = []
print('Kiểm tra checkpoint cho temporal-residual diagnostics:')
for config in temporal_models:
    model_path = Path(config['model_path'])
    if model_path.exists():
        valid_temporal_models.append(config)
        print(f"  FOUND   {config['name']}: {model_path}")
    else:
        print(f"  MISSING {config['name']}: {model_path}")

if not valid_temporal_models:
    raise FileNotFoundError('Không có checkpoint hợp lệ để phân tích.')

temporal_models_path = TEMPORAL_RESIDUAL_OUTPUT_DIR / 'models_to_analyze.json'
with temporal_models_path.open('w', encoding='utf-8') as f:
    json.dump(valid_temporal_models, f, indent=2, ensure_ascii=False)

temporal_residual_cmd = [
    'python', 'ucf_analyze_temporal_residuals.py',
    '--models-json', str(temporal_models_path),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--top-k-ratio', '0.15',
    '--local-radius', '2',
    '--output-dir', str(TEMPORAL_RESIDUAL_OUTPUT_DIR),
]

print('\nĐang chạy:', ' '.join(shlex.quote(part) for part in temporal_residual_cmd))
print('Không encode lại video; chỉ tính toán trên CLIP feature hiện có.')
run_command(temporal_residual_cmd)

print('\nKết quả temporal-residual diagnostics:', TEMPORAL_RESIDUAL_OUTPUT_DIR)
print('Các file chính:')
for filename in [
    'temporal_residual_group_summary.csv',
    'temporal_residual_separation.csv',
    'temporal_residual_per_class.csv',
    'temporal_residual_score_correlation.csv',
    'analysis_metadata.json',
]:
    print(' ', TEMPORAL_RESIDUAL_OUTPUT_DIR / filename)


## 29. Kiểm Chứng Counterfactual Segment Masking

Cell này thay từng chuỗi snippet có score cao bằng feature nội suy hoặc feature trung bình của ngữ cảnh, sau đó chạy lại model để đo mức thay đổi output. Phân tích cũng kiểm tra các chuỗi anomaly bị bỏ sót bằng ground truth, nhưng ground truth chỉ dùng để đặt tên nhóm sau inference. Không tính AUC/AP và không thay đổi checkpoint.

In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')
if 'ANALYZE_RESULTS_DIR' not in globals():
    ANALYZE_RESULTS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze'

os.chdir(SRC_DIR)
analyzer_path = SRC_DIR / 'ucf_analyze_counterfactual_masking.py'
if not analyzer_path.exists():
    raise FileNotFoundError(
        f'Không tìm thấy {analyzer_path}. Hãy upload file analyzer mới vào VadCLIP/src.'
    )
if not FEATURE_ROOT.exists():
    raise FileNotFoundError(
        f'Không tìm thấy feature root: {FEATURE_ROOT}. Hãy chạy cell giải nén UCFClipFeatures trước.'
    )

COUNTERFACTUAL_OUTPUT_DIR = ANALYZE_RESULTS_DIR / 'counterfactual_masking'
COUNTERFACTUAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Có thể thay danh sách này bằng cấu hình checkpoint khác ở những lần phân tích sau.
if 'valid_temporal_models' in globals() and valid_temporal_models:
    counterfactual_models = valid_temporal_models
elif 'valid_failure_models' in globals() and valid_failure_models:
    counterfactual_models = valid_failure_models
elif 'valid_visual_models' in globals() and valid_visual_models:
    counterfactual_models = valid_visual_models
elif 'available_models' in globals() and available_models:
    counterfactual_models = available_models
elif 'MODELS_TO_ANALYZE' in globals():
    counterfactual_models = MODELS_TO_ANALYZE
else:
    config_candidates = [
        ANALYZE_RESULTS_DIR / 'temporal_residual_diagnostics' / 'models_to_analyze.json',
        ANALYZE_RESULTS_DIR / 'failure_source_diagnostics' / 'models_to_analyze.json',
        ANALYZE_RESULTS_DIR / 'visual_feature_diagnostics' / 'models_to_analyze.json',
        ANALYZE_RESULTS_DIR / 'snippet_topology' / 'models_to_analyze.json',
    ]
    config_path = next((path for path in config_candidates if path.exists()), None)
    if config_path is None:
        raise FileNotFoundError(
            'Không tìm thấy cấu hình checkpoint. Hãy chạy một cell diagnostics trước hoặc khai báo MODELS_TO_ANALYZE.'
        )
    with config_path.open('r', encoding='utf-8') as f:
        counterfactual_models = json.load(f)

valid_counterfactual_models = []
print('Kiểm tra checkpoint cho counterfactual masking:')
for config in counterfactual_models:
    model_path = Path(config['model_path'])
    if model_path.exists():
        valid_counterfactual_models.append(config)
        print(f"  FOUND   {config['name']}: {model_path}")
    else:
        print(f"  MISSING {config['name']}: {model_path}")

if not valid_counterfactual_models:
    raise FileNotFoundError('Không có checkpoint hợp lệ để phân tích.')

counterfactual_models_path = COUNTERFACTUAL_OUTPUT_DIR / 'models_to_analyze.json'
with counterfactual_models_path.open('w', encoding='utf-8') as f:
    json.dump(valid_counterfactual_models, f, indent=2, ensure_ascii=False)

counterfactual_cmd = [
    'python', 'ucf_analyze_counterfactual_masking.py',
    '--models-json', str(counterfactual_models_path),
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--top-k-ratio', '0.15',
    '--mask-methods', 'linear,context_mean',
    '--context-radius', '3',
    '--neighbor-radius', '4',
    '--max-segments-per-group', '1',
    '--min-run-length', '1',
    '--include-missed-anomaly',
    '--output-dir', str(COUNTERFACTUAL_OUTPUT_DIR),
]

print('\nĐang chạy:', ' '.join(shlex.quote(part) for part in counterfactual_cmd))
print('Mỗi segment được thay riêng; model và checkpoint không bị chỉnh sửa.')
print('Không tính AUC/AP, chỉ đo thay đổi output trước và sau masking.')
run_command(counterfactual_cmd)

print('\nKết quả counterfactual masking:', COUNTERFACTUAL_OUTPUT_DIR)
print('Các file chính:')
for filename in [
    'counterfactual_response_summary.csv',
    'counterfactual_response_per_class.csv',
    'counterfactual_segment_details.csv',
    'analysis_metadata.json',
]:
    print(' ', COUNTERFACTUAL_OUTPUT_DIR / filename)


## 30. Train VadCLIP Với Learnable Temporal Proposals

Cell này train một kiến trúc mới từ scratch. Encoder visual và text prompt gốc của VadCLIP được giữ lại, nhưng snippet top-k MIL được thay bằng 8 temporal proposal query tự học tâm và độ dài segment. Mỗi epoch được evaluate một lần; training dừng sớm nếu classifier AUC không cải thiện trong 3 epoch. Không dùng description, prototype prompt hoặc các regularizer top-k trước đó.

In [ ]:
import csv
import os
import shlex
import subprocess
from pathlib import Path

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')

os.chdir(SRC_DIR)
required_scripts = [
    'model_temporal_proposal.py',
    'ucf_option_temporal_proposal.py',
    'ucf_train_temporal_proposal.py',
]
for script_name in required_scripts:
    script_path = SRC_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f'Không tìm thấy file mới: {script_path}')

proposal_feature_root = (
    Path(LOCAL_FEATURE_ROOT)
    if 'LOCAL_FEATURE_ROOT' in globals() and Path(LOCAL_FEATURE_ROOT).exists()
    else Path(FEATURE_ROOT)
)
if not proposal_feature_root.exists():
    raise FileNotFoundError(
        f'Không tìm thấy feature root: {proposal_feature_root}. Hãy chạy cell giải nén feature trước.'
    )
print('Temporal proposal feature root:', proposal_feature_root)

def write_proposal_absolute_list(relative_csv, output_csv):
    rows = []
    missing = []
    with Path(relative_csv).open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        for row in reader:
            item = dict(row)
            feature_path = (proposal_feature_root / item['path']).resolve()
            if not feature_path.exists():
                missing.append(str(feature_path))
            item['path'] = str(feature_path)
            rows.append(item)
    if missing:
        report = Path(output_csv).with_name(Path(output_csv).stem + '_missing.txt')
        report.parent.mkdir(parents=True, exist_ok=True)
        report.write_text('\n'.join(missing), encoding='utf-8')
        raise FileNotFoundError(
            f'Có {len(missing)} feature bị thiếu. Xem danh sách tại {report}.'
        )
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_csv).open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Wrote {len(rows)} rows to {output_csv}')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
PROPOSAL_LIST_DIR = PROJECT_ROOT / 'code' / 'lists_proposal_colab'
proposal_train_list = PROPOSAL_LIST_DIR / 'train.csv'
proposal_test_list = PROPOSAL_LIST_DIR / 'test.csv'
write_proposal_absolute_list(LIST_DIR / 'ucf_CLIP_rgb_relative.csv', proposal_train_list)
write_proposal_absolute_list(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', proposal_test_list)

PROPOSAL_MODEL = 'model/model_ucf_proposal_scratch.pth'
PROPOSAL_BEST = 'model/checkpoint_proposal_scratch.pth'
PROPOSAL_CURRENT = 'model/model_cur_proposal_scratch.pth'
PROPOSAL_EPOCH_DIR = 'model/epoch_checkpoints_proposal_scratch'
PROPOSAL_RESULTS_DIR = PROJECT_ROOT / 'code' / 'results' / 'evaluate' / 'temporal_proposal_scratch'

proposal_train_cmd = [
    'python', 'ucf_train_temporal_proposal.py',
    '--feature-root', str(proposal_feature_root),
    '--train-list', str(proposal_train_list),
    '--test-list', str(proposal_test_list),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--num-proposals', '8',
    '--proposal-heads', '4',
    '--proposal-min-width', '0.02',
    '--proposal-max-width', '0.50',
    '--proposal-initial-width', '0.15',
    '--mask-temperature', '0.02',
    '--context-scale', '0.25',
    '--lambda-normal', '0.50',
    '--lambda-class', '1.00',
    '--lambda-width', '0.0',
    '--lambda-coverage', '0.05',
    '--proposal-min-coverage', '0.05',
    '--proposal-max-coverage', '0.50',
    '--lambda-overlap', '0.05',
    '--lambda-boundary', '0.01',
    '--boundary-warmup-epochs', '1',
    '--lr', '2e-5',
    '--max-epoch', '10',
    '--early-stopping-patience', '3',
    '--batch-size', '64',
    '--scheduler-milestones', '4', '8',
    '--num-workers', '2',
    '--pin-memory', 'true',
    '--output-model-path', PROPOSAL_MODEL,
    '--checkpoint-path', PROPOSAL_BEST,
    '--save-cur-path', PROPOSAL_CURRENT,
    '--epoch-checkpoint-dir', PROPOSAL_EPOCH_DIR,
    '--results-dir', str(PROPOSAL_RESULTS_DIR),
]

print('\nTraining learnable temporal proposal VadCLIP from scratch')
print('Running:', ' '.join(shlex.quote(part) for part in proposal_train_cmd))
run_command(proposal_train_cmd)

print('\nTemporal proposal outputs:')
print('  best model:', SRC_DIR / PROPOSAL_MODEL)
print('  epoch checkpoints:', SRC_DIR / PROPOSAL_EPOCH_DIR)
print('  training/evaluation history:', PROPOSAL_RESULTS_DIR)


## 31. Train VadCLIP với Hard-Negative Transplantation và Evidence Erasing

Cell này train VadCLIP từ scratch với hai cơ chế bổ sung. Thứ nhất, một đoạn có score cao từ video Normal được ghép vào video abnormal. Vùng đích phải thấp ở cả logits1 và logits2, nằm ngoài bán kính local attention của mọi high-score snippet, đồng thời có context trái/phải giống đoạn nguồn nhất. Biên ghép được cross-fade và loss chỉ tính trên lõi Normal thuần. Thứ hai, đoạn bằng chứng có score cao nhất trong video abnormal được che bằng context lân cận, sau đó vẫn giữ supervision cấp video. Hai loss mới chỉ bật sau warmup; không dùng top-k làm pseudo-label positive. Cell cũng theo dõi chính xác top-k mà hai MIL loss chọn qua từng epoch để kiểm tra evidence collapse.

In [ ]:
import csv
import os
import shlex
import subprocess
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')

os.chdir(SRC_DIR)
TRAIN_SCRIPT = SRC_DIR / 'ucf_train.py'
if not TRAIN_SCRIPT.exists():
    raise FileNotFoundError(f'Không tìm thấy train script: {TRAIN_SCRIPT}')

augmentation_feature_root = (
    Path(LOCAL_FEATURE_ROOT)
    if 'LOCAL_FEATURE_ROOT' in globals() and Path(LOCAL_FEATURE_ROOT).exists()
    else Path(FEATURE_ROOT)
)
if not augmentation_feature_root.exists():
    raise FileNotFoundError(
        f'Không tìm thấy feature root: {augmentation_feature_root}. ' 
        'Hãy chạy cell giải nén feature trước.'
    )
print('Feature root:', augmentation_feature_root)

def write_augmentation_absolute_list(relative_csv, output_csv):
    rows = []
    missing = []
    with Path(relative_csv).open('r', encoding='utf-8') as source_file:
        reader = csv.DictReader(source_file)
        fieldnames = reader.fieldnames
        for row in reader:
            item = dict(row)
            feature_path = (augmentation_feature_root / item['path']).resolve()
            if not feature_path.exists():
                missing.append(str(feature_path))
            item['path'] = str(feature_path)
            rows.append(item)
    if missing:
        report_path = Path(output_csv).with_name(Path(output_csv).stem + '_missing.txt')
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text('\n'.join(missing), encoding='utf-8')
        raise FileNotFoundError(
            f'Có {len(missing)} feature bị thiếu. Xem {report_path}.'
        )
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_csv).open('w', encoding='utf-8', newline='') as target_file:
        writer = csv.DictWriter(target_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Wrote {len(rows)} rows to {output_csv}')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
AUGMENTATION_LIST_DIR = PROJECT_ROOT / 'code' / 'lists_transplant_erasing_colab'
augmentation_train_list = AUGMENTATION_LIST_DIR / 'train.csv'
augmentation_test_list = AUGMENTATION_LIST_DIR / 'test.csv'
write_augmentation_absolute_list(
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv', augmentation_train_list
)
write_augmentation_absolute_list(
    LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', augmentation_test_list
)

AUGMENTATION_RESULTS_DIR = (
    PROJECT_ROOT / 'code' / 'results' / 'evaluate' / 'transplant_erasing_scratch'
)
AUGMENTATION_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TOPK_TRACKING_DIR = (
    PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'topk_tracking_transplant_erasing'
)
AUGMENTATION_MODEL = 'model/model_ucf_transplant_erasing_scratch.pth'
AUGMENTATION_CHECKPOINT = 'model/checkpoint_transplant_erasing_scratch.pth'
AUGMENTATION_CURRENT = 'model/model_cur_transplant_erasing_scratch.pth'
AUGMENTATION_EPOCH_DIR = 'model/epoch_checkpoints_transplant_erasing_scratch'

train_cmd = [
    'python', 'ucf_train.py',
    '--training-mode', 'transplant_erasing_epoch_eval',
    '--train-list', str(augmentation_train_list),
    '--test-list', str(augmentation_test_list),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lr', '1e-5',
    '--max-epoch', '10',
    '--batch-size', '64',
    '--seed', '234',
    '--lambda-transplant', '0.05',
    '--lambda-evidence-erasing', '0.10',
    '--augmentation-warmup-epochs', '2',
    '--augmentation-min-segment-length', '4',
    '--augmentation-max-segment-length', '12',
    '--transplant-score-branch', 'classifier',
    '--transplant-context-width', '8',
    '--transplant-high-exclusion-ratio', '0.15',
    '--transplant-boundary-blend-width', '1',
    '--track-topk',
    '--topk-tracking-output-dir', str(TOPK_TRACKING_DIR),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
    '--early-stopping-patience', '3',
    '--model-path', AUGMENTATION_MODEL,
    '--checkpoint-path', AUGMENTATION_CHECKPOINT,
    '--save-cur-path', AUGMENTATION_CURRENT,
    '--epoch-checkpoint-dir', AUGMENTATION_EPOCH_DIR,
]

print('Train VadCLIP from scratch với transplantation + evidence erasing')
print('Warmup: 2 epoch; loss mới bằng 0 trong epoch 1-2')
print('Running:', ' '.join(shlex.quote(part) for part in train_cmd))
log_path = AUGMENTATION_RESULTS_DIR / 'training_console.txt'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for output_line in process.stdout:
        print(output_line, end='')
        log_file.write(output_line)
    return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, train_cmd)

print('\nKết quả:')
print('  best model:', SRC_DIR / AUGMENTATION_MODEL)
print('  epoch checkpoints:', SRC_DIR / AUGMENTATION_EPOCH_DIR)
print('  console/evaluation log:', log_path)
print('  top-k tracking:', TOPK_TRACKING_DIR)


## 32. Đối Chiếu MIL Top-k Qua Các Checkpoint Với Temporal Ground Truth

Cell này chỉ chạy inference với `torch.no_grad()`, không dùng test set để train và không cập nhật model. Với từng checkpoint, top-k mà MIL sẽ chọn được đối chiếu với temporal ground truth để đo độ chính xác của top-k, tỷ lệ anomaly được bao phủ, anomaly liên tục bị bỏ sót và mức ổn định của top-k giữa hai epoch.

Nếu thư mục checkpoint theo epoch chưa tồn tại, cell vẫn phân tích model cuối nhưng không thể đo quá trình khóa evidence qua nhiều epoch. Để có đầy đủ trajectory, chạy lại cell normal top-k sau khi upload phiên bản mới của `ucf_train.py` và `ucf_option.py`; cell đó hiện lưu riêng model của từng epoch.


In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

ANALYZER = SRC_DIR / 'ucf_analyze_test_topk_checkpoints.py'
MODEL_TYPE = 'baseline'
CHECKPOINT_DIR = SRC_DIR / 'model' / 'epoch_checkpoints_paper_normal_topk_scratch'
FINAL_MODEL = SRC_DIR / 'model' / 'model_ucf_paper_normal_topk_scratch.pth'
OUTPUT_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'test_topk_normal_topk'
TEST_LIST = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv'
GT_PATH = PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_ucf.npy'

feature_candidates = [
    Path('/content/UCFClipFeatures'),
    Path(globals().get('LOCAL_FEATURE_ROOT', '/content/UCFClipFeatures')),
    Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures')),
]
FEATURE_ROOT_FOR_ANALYSIS = next((path for path in feature_candidates if path.exists()), None)
if FEATURE_ROOT_FOR_ANALYSIS is None:
    raise FileNotFoundError('Chưa tìm thấy UCFClipFeatures đã extract trong /content hoặc FEATURE_ROOT hiện tại.')
for required in [ANALYZER, TEST_LIST, GT_PATH]:
    if not required.exists():
        raise FileNotFoundError(required)

epoch_checkpoints = sorted(CHECKPOINT_DIR.glob('*.pth')) if CHECKPOINT_DIR.exists() else []
analyze_cmd = [
    sys.executable, str(ANALYZER),
    '--model-type', MODEL_TYPE,
    '--feature-root', str(FEATURE_ROOT_FOR_ANALYSIS),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(GT_PATH),
    '--output-dir', str(OUTPUT_DIR),
]
if epoch_checkpoints:
    analyze_cmd.extend(['--checkpoint-dir', str(CHECKPOINT_DIR)])
    print(f'Phân tích trajectory của {len(epoch_checkpoints)} checkpoint theo epoch.')
elif FINAL_MODEL.exists():
    analyze_cmd.extend(['--checkpoint-path', str(FINAL_MODEL)])
    print('Chỉ tìm thấy model cuối. Kết quả sẽ đo top-k/GT tại một thời điểm, không đo được khóa evidence qua epoch.')
else:
    raise FileNotFoundError(f'Không tìm thấy checkpoint trong {CHECKPOINT_DIR} hoặc model cuối {FINAL_MODEL}')

# Khi phân tích multi-prompt, đổi MODEL_TYPE và khai báo các đối số prompt tương ứng tại đây.
# analyze_cmd.extend([
#     '--prototype-json', str(PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'),
#     '--prompt-weights-json', str(PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'),
#     '--prototype-source', 'manual', '--prompt-mode', 'manual_caption',
# ])

print('Feature root:', FEATURE_ROOT_FOR_ANALYSIS)
print('Output:', OUTPUT_DIR)
print('Working directory: /content')
print('Running:', ' '.join(shlex.quote(part) for part in analyze_cmd))
subprocess.run(analyze_cmd, cwd='/content', check=True)


## 33. Train VadCLIP từ scratch với hai classifier head và diversity loss

Cell này giữ nguyên encoder và nhánh class-text của VadCLIP, thay classifier nhị phân duy nhất bằng hai head. Hai raw logits được lấy trung bình để tạo `logits1`; diversity chỉ áp dụng cho video bất thường sau 2 epoch warmup. Model được đánh giá sau mỗi epoch và lưu tracker riêng cho aggregate, head 1 và head 2.


In [ ]:
import csv
import shlex
import subprocess
import sys
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'

feature_candidates = [
    Path('/content/UCFClipFeatures'),
    Path(globals().get('LOCAL_FEATURE_ROOT', '/content/UCFClipFeatures')),
    Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures')),
]
MULTI_HEAD_FEATURE_ROOT = next((path for path in feature_candidates if path.exists()), None)
if MULTI_HEAD_FEATURE_ROOT is None:
    raise FileNotFoundError('Chưa tìm thấy UCFClipFeatures đã giải nén trong /content.')

required_scripts = [
    SRC_DIR / 'ucf_train.py',
    SRC_DIR / 'ucf_test.py',
    SRC_DIR / 'model.py',
    SRC_DIR / 'model_multi_classifier.py',
    SRC_DIR / 'ucf_analyze_multi_classifier_heads.py',
]
for required_path in required_scripts:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

def write_multi_head_absolute_list(relative_csv, output_csv):
    rows = []
    missing = []
    with Path(relative_csv).open('r', encoding='utf-8') as input_file:
        reader = csv.DictReader(input_file)
        fieldnames = reader.fieldnames
        for row in reader:
            item = dict(row)
            feature_path = (MULTI_HEAD_FEATURE_ROOT / item['path']).resolve()
            if not feature_path.exists():
                missing.append(str(feature_path))
            item['path'] = str(feature_path)
            rows.append(item)
    if missing:
        raise FileNotFoundError(f'Thiếu {len(missing)} feature; ví dụ: {missing[:3]}')
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_csv).open('w', encoding='utf-8', newline='') as output_file:
        writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Đã ghi {len(rows)} dòng vào {output_csv}')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
ABS_LIST_DIR = PROJECT_ROOT / 'code' / 'results' / 'lists' / 'multi_classifier'
TRAIN_LIST = ABS_LIST_DIR / 'train.csv'
TEST_LIST = ABS_LIST_DIR / 'test.csv'
write_multi_head_absolute_list(LIST_DIR / 'ucf_CLIP_rgb_relative.csv', TRAIN_LIST)
write_multi_head_absolute_list(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', TEST_LIST)

NUM_HEADS = 2
LAMBDA_DIVERSITY = 0.01
LAMBDA_BALANCE = 0.001
DIVERSITY_TEMPERATURE = 1.0
DIVERSITY_WARMUP_EPOCHS = 2
DIVERSITY_RAMP_EPOCHS = 2
AGGREGATE_CLS_WEIGHT = 0.5

MODEL_PATH = SRC_DIR / 'model' / 'model_ucf_multi_classifier_diversity_scratch.pth'
CHECKPOINT_PATH = SRC_DIR / 'model' / 'checkpoint_multi_classifier_diversity_scratch.pth'
CURRENT_PATH = SRC_DIR / 'model' / 'model_cur_multi_classifier_diversity_scratch.pth'
EPOCH_DIR = SRC_DIR / 'model' / 'epoch_checkpoints_multi_classifier_diversity_scratch'
RESULT_DIR = PROJECT_ROOT / 'code' / 'results' / 'train' / 'multi_classifier_diversity'
TRACKING_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'topk_multi_classifier_diversity'
HEAD_DIAGNOSTICS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'multi_classifier_head_diagnostics'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable, str(SRC_DIR / 'ucf_train.py'),
    '--training-mode', 'multi_classifier_epoch_eval',
    '--train-list', str(TRAIN_LIST),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
    '--lr', '1e-5',
    '--max-epoch', '10',
    '--batch-size', '64',
    '--seed', '234',
    '--num-classifier-heads', str(NUM_HEADS),
    '--lambda-head-diversity', str(LAMBDA_DIVERSITY),
    '--lambda-head-balance', str(LAMBDA_BALANCE),
    '--head-diversity-temperature', str(DIVERSITY_TEMPERATURE),
    '--head-diversity-warmup-epochs', str(DIVERSITY_WARMUP_EPOCHS),
    '--head-diversity-ramp-epochs', str(DIVERSITY_RAMP_EPOCHS),
    '--multi-head-cls-aggregate-weight', str(AGGREGATE_CLS_WEIGHT),
    '--early-stopping-patience', '3',
    '--model-path', str(MODEL_PATH),
    '--checkpoint-path', str(CHECKPOINT_PATH),
    '--save-cur-path', str(CURRENT_PATH),
    '--epoch-checkpoint-dir', str(EPOCH_DIR),
    '--track-topk',
    '--topk-tracking-output-dir', str(TRACKING_DIR),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
]

print('Feature root:', MULTI_HEAD_FEATURE_ROOT)
print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
log_path = RESULT_DIR / 'training_console.txt'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        train_cmd, cwd=str(SRC_DIR), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for output_line in process.stdout:
        print(output_line, end='')
        log_file.write(output_line)
    return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, train_cmd)

test_cmd = [
    sys.executable, str(SRC_DIR / 'ucf_test.py'),
    '--model-type', 'multi_classifier',
    '--num-classifier-heads', str(NUM_HEADS),
    '--model-path', str(MODEL_PATH),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
]
print('Đánh giá model tốt nhất:', ' '.join(shlex.quote(part) for part in test_cmd))
subprocess.run(test_cmd, cwd=str(SRC_DIR), check=True)

head_diagnostics_cmd = [
    sys.executable, str(SRC_DIR / 'ucf_analyze_multi_classifier_heads.py'),
    '--model-path', str(MODEL_PATH),
    '--num-classifier-heads', str(NUM_HEADS),
    '--feature-root', str(MULTI_HEAD_FEATURE_ROOT),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--output-dir', str(HEAD_DIAGNOSTICS_DIR),
]
print('Analyzing individual classifier heads:', ' '.join(shlex.quote(part) for part in head_diagnostics_cmd))
subprocess.run(head_diagnostics_cmd, cwd=str(SRC_DIR), check=True)

print('Model tốt nhất:', MODEL_PATH)
print('Checkpoint theo epoch:', EPOCH_DIR)
print('Training log:', log_path)
print('Top-k aggregate:', TRACKING_DIR)
print('Top-k từng head:', TRACKING_DIR / 'head_1', 'và', TRACKING_DIR / 'head_2')


## 34. Train VadCLIP từ scratch với residual adapter riêng cho từng classifier head

Cell này triển khai giai đoạn 1 của thí nghiệm multi-head: backbone và nhánh class-text vẫn dùng chung, nhưng mỗi classifier head có một residual adapter `512 -> 128 -> 512` riêng. Loss, cách lấy trung bình raw logits và lịch warmup diversity được giữ giống thí nghiệm multi-head cũ để đo riêng tác động của việc tách representation.


In [ ]:
import csv
import shlex
import subprocess
import sys
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'

feature_candidates = [
    Path('/content/UCFClipFeatures'),
    Path(globals().get('LOCAL_FEATURE_ROOT', '/content/UCFClipFeatures')),
    Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures')),
]
ADAPTER_FEATURE_ROOT = next((path for path in feature_candidates if path.exists()), None)
if ADAPTER_FEATURE_ROOT is None:
    raise FileNotFoundError('Chưa tìm thấy UCFClipFeatures đã giải nén trong /content.')

required_scripts = [
    SRC_DIR / 'ucf_train.py',
    SRC_DIR / 'ucf_test.py',
    SRC_DIR / 'model.py',
    SRC_DIR / 'model_multi_classifier.py',
    SRC_DIR / 'ucf_option.py',
    SRC_DIR / 'ucf_analyze_multi_classifier_heads.py',
]
for required_path in required_scripts:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

def write_adapter_absolute_list(relative_csv, output_csv):
    rows = []
    missing = []
    with Path(relative_csv).open('r', encoding='utf-8') as input_file:
        reader = csv.DictReader(input_file)
        fieldnames = reader.fieldnames
        for row in reader:
            item = dict(row)
            feature_path = (ADAPTER_FEATURE_ROOT / item['path']).resolve()
            if not feature_path.exists():
                missing.append(str(feature_path))
            item['path'] = str(feature_path)
            rows.append(item)
    if missing:
        raise FileNotFoundError(f'Thiếu {len(missing)} feature; ví dụ: {missing[:3]}')
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    with Path(output_csv).open('w', encoding='utf-8', newline='') as output_file:
        writer = csv.DictWriter(output_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Đã ghi {len(rows)} dòng vào {output_csv}')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
ABS_LIST_DIR = PROJECT_ROOT / 'code' / 'results' / 'lists' / 'adapter_multi_classifier'
TRAIN_LIST = ABS_LIST_DIR / 'train.csv'
TEST_LIST = ABS_LIST_DIR / 'test.csv'
write_adapter_absolute_list(LIST_DIR / 'ucf_CLIP_rgb_relative.csv', TRAIN_LIST)
write_adapter_absolute_list(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv', TEST_LIST)

NUM_HEADS = 2
ADAPTER_BOTTLENECK = 128
ADAPTER_DROPOUT = 0.1
ADAPTER_RESIDUAL_SCALE = 0.1
LAMBDA_DIVERSITY = 0.01
LAMBDA_BALANCE = 0.001
DIVERSITY_TEMPERATURE = 1.0
DIVERSITY_WARMUP_EPOCHS = 2
DIVERSITY_RAMP_EPOCHS = 2
AGGREGATE_CLS_WEIGHT = 0.5

MODEL_PATH = SRC_DIR / 'model' / 'model_ucf_adapter_multi_classifier_scratch.pth'
CHECKPOINT_PATH = SRC_DIR / 'model' / 'checkpoint_adapter_multi_classifier_scratch.pth'
CURRENT_PATH = SRC_DIR / 'model' / 'model_cur_adapter_multi_classifier_scratch.pth'
EPOCH_DIR = SRC_DIR / 'model' / 'epoch_checkpoints_adapter_multi_classifier_scratch'
RESULT_DIR = PROJECT_ROOT / 'code' / 'results' / 'train' / 'adapter_multi_classifier'
TRACKING_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'topk_adapter_multi_classifier'
HEAD_DIAGNOSTICS_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'adapter_multi_classifier_heads'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

adapter_args = [
    '--num-classifier-heads', str(NUM_HEADS),
    '--head-adapter-bottleneck', str(ADAPTER_BOTTLENECK),
    '--head-adapter-dropout', str(ADAPTER_DROPOUT),
    '--head-adapter-residual-scale', str(ADAPTER_RESIDUAL_SCALE),
]
train_cmd = [
    sys.executable, str(SRC_DIR / 'ucf_train.py'),
    '--training-mode', 'adapter_multi_classifier_epoch_eval',
    '--train-list', str(TRAIN_LIST),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
    '--lr', '1e-5',
    '--max-epoch', '10',
    '--batch-size', '64',
    '--seed', '234',
    *adapter_args,
    '--lambda-head-diversity', str(LAMBDA_DIVERSITY),
    '--lambda-head-balance', str(LAMBDA_BALANCE),
    '--head-diversity-temperature', str(DIVERSITY_TEMPERATURE),
    '--head-diversity-warmup-epochs', str(DIVERSITY_WARMUP_EPOCHS),
    '--head-diversity-ramp-epochs', str(DIVERSITY_RAMP_EPOCHS),
    '--multi-head-cls-aggregate-weight', str(AGGREGATE_CLS_WEIGHT),
    '--early-stopping-patience', '3',
    '--model-path', str(MODEL_PATH),
    '--checkpoint-path', str(CHECKPOINT_PATH),
    '--save-cur-path', str(CURRENT_PATH),
    '--epoch-checkpoint-dir', str(EPOCH_DIR),
    '--track-topk',
    '--topk-tracking-output-dir', str(TRACKING_DIR),
    '--topk-tracking-branches', 'both',
    '--topk-tracking-scope', 'abnormal',
]

print('Feature root:', ADAPTER_FEATURE_ROOT)
print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
log_path = RESULT_DIR / 'training_console.txt'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        train_cmd, cwd=str(SRC_DIR), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for output_line in process.stdout:
        print(output_line, end='')
        log_file.write(output_line)
    return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, train_cmd)

test_cmd = [
    sys.executable, str(SRC_DIR / 'ucf_test.py'),
    '--model-type', 'adapter_multi_classifier',
    *adapter_args,
    '--model-path', str(MODEL_PATH),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
]
print('Đánh giá model tốt nhất:', ' '.join(shlex.quote(part) for part in test_cmd))
subprocess.run(test_cmd, cwd=str(SRC_DIR), check=True)

head_diagnostics_cmd = [
    sys.executable, str(SRC_DIR / 'ucf_analyze_multi_classifier_heads.py'),
    '--model-type', 'adapter_multi_classifier',
    *adapter_args,
    '--model-path', str(MODEL_PATH),
    '--feature-root', str(ADAPTER_FEATURE_ROOT),
    '--test-list', str(TEST_LIST),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--output-dir', str(HEAD_DIAGNOSTICS_DIR),
]
print('Phân tích từng classifier head:', ' '.join(shlex.quote(part) for part in head_diagnostics_cmd))
subprocess.run(head_diagnostics_cmd, cwd=str(SRC_DIR), check=True)

print('Model tốt nhất:', MODEL_PATH)
print('Checkpoint theo epoch:', EPOCH_DIR)
print('Training log:', log_path)
print('Top-k tracking:', TRACKING_DIR)
print('Head diagnostics:', HEAD_DIAGNOSTICS_DIR)


## 35. Phân tích Normal visual bank toàn diện

Cell này chỉ phân tích baseline tự train, không training và không thay đổi loss. Toàn bộ 8.000 Normal chunks được quan sát; feature được lấy mẫu cân bằng theo video rồi dùng để so sánh bank 32 centroid, 256 centroid và hybrid 256 centroid + 2.048 exemplar hiếm.


In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'

feature_candidates = [
    Path('/content/UCFClipFeatures'),
    Path(globals().get('LOCAL_FEATURE_ROOT', '/content/UCFClipFeatures')),
    Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures')),
]
NORMAL_BANK_FEATURE_ROOT = next(
    (path for path in feature_candidates if path.exists()), None
)
if NORMAL_BANK_FEATURE_ROOT is None:
    raise FileNotFoundError(
        'Chưa tìm thấy /content/UCFClipFeatures. Hãy chạy cell giải nén feature trước.'
    )

NORMAL_BANK_SCRIPT = SRC_DIR / 'ucf_analyze_normal_bank.py'
baseline_candidates = [
    SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth',
    PROJECT_ROOT / 'model_ucf.pth',
]
BASELINE_MODEL = next(
    (path for path in baseline_candidates if path.exists()), None
)
if BASELINE_MODEL is None:
    raise FileNotFoundError(
        'Không tìm thấy model_ucf_paper_scratch.pth hoặc model_ucf.pth.'
    )
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
NORMAL_BANK_OUTPUT_DIR = (
    PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'normal_bank'
)

required_paths = [
    NORMAL_BANK_SCRIPT,
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
    LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
    LIST_DIR / 'gt_ucf.npy',
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

NORMAL_BANK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
normal_bank_cmd = [
    sys.executable, str(NORMAL_BANK_SCRIPT),
    '--model-path', str(BASELINE_MODEL),
    '--feature-root', str(NORMAL_BANK_FEATURE_ROOT),
    '--train-list', str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
    '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--output-dir', str(NORMAL_BANK_OUTPUT_DIR),
    '--snippets-per-chunk', '8',
    '--max-snippets-per-video', '80',
    '--heldout-ratio', '0.2',
    '--old-style-prototypes', '32',
    '--expanded-prototypes', '256',
    '--rare-exemplars', '2048',
    '--max-exemplars-per-video', '8',
    '--similarity-neighbors', '5',
    '--top-k-ratio', '0.15',
    '--rescore-gammas', '0,0.25,0.5,1',
    '--normal-batch-size', '16',
    '--seed', '42',
]

print('Normal bank feature root:', NORMAL_BANK_FEATURE_ROOT)
print('Baseline checkpoint:', BASELINE_MODEL)
if BASELINE_MODEL == PROJECT_ROOT / 'model_ucf.pth':
    print('Lưu ý: đang phân tích checkpoint model_ucf.pth của VadCLIP, không phải baseline tự train.')
print('Output:', NORMAL_BANK_OUTPUT_DIR)
print('Running:', ' '.join(shlex.quote(part) for part in normal_bank_cmd))
log_path = NORMAL_BANK_OUTPUT_DIR / 'analysis_console.txt'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        normal_bank_cmd,
        cwd=str(SRC_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for output_line in process.stdout:
        print(output_line, end='')
        log_file.write(output_line)
    return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, normal_bank_cmd)

print('\nCác file kết quả:')
for result_path in sorted(NORMAL_BANK_OUTPUT_DIR.iterdir()):
    print(' ', result_path.name)


## 36. Phân tích Cross-Video Class Evidence

Cell này chỉ phân tích, không train và không cập nhật model. Mục tiêu là kiểm tra xem các snippet `true-high`, `false-high` và `missed-anomaly` có thực sự nằm trong những vùng feature khác nhau giữa nhiều video hay không.

Bank được xây dựng từ train split bằng cả snippet có classifier score cao và snippet lấy đều theo thời gian, sau đó cân bằng theo video. Temporal ground truth của test split chỉ dùng để đặt tên các nhóm phục vụ đánh giá; nó không tham gia xây bank. Kết quả được tính riêng trên CLIP feature đầu vào và feature thực sự đi vào classifier của VadCLIP.


In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'

feature_candidates = [
    Path(globals().get('LOCAL_FEATURE_ROOT', '/content/UCFClipFeatures')),
    Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures')),
]
CROSS_VIDEO_FEATURE_ROOT = next((path for path in feature_candidates if path.exists()), None)
if CROSS_VIDEO_FEATURE_ROOT is None:
    raise FileNotFoundError('Chưa tìm thấy UCFClipFeatures đã giải nén trong /content. Hãy chạy cell giải nén trước.')

model_candidates = [
    Path(globals()['CROSS_VIDEO_MODEL_OVERRIDE']) if globals().get('CROSS_VIDEO_MODEL_OVERRIDE') else None,
    SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth',
    PROJECT_ROOT / 'model_ucf.pth',
]
model_candidates = [path for path in model_candidates if path is not None]
CROSS_VIDEO_MODEL = next((path for path in model_candidates if path.exists()), None)
if CROSS_VIDEO_MODEL is None:
    raise FileNotFoundError('Không tìm thấy baseline tự train hoặc model_ucf.pth.')

LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
CROSS_VIDEO_SCRIPT = SRC_DIR / 'ucf_analyze_cross_video_evidence.py'
CROSS_VIDEO_OUTPUT_DIR = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'cross_video_evidence'
required_paths = [
    CROSS_VIDEO_SCRIPT,
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
    LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
    LIST_DIR / 'gt_ucf.npy',
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

CROSS_VIDEO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cross_video_cmd = [
    sys.executable, str(CROSS_VIDEO_SCRIPT),
    '--model-path', str(CROSS_VIDEO_MODEL),
    '--feature-root', str(CROSS_VIDEO_FEATURE_ROOT),
    '--train-list', str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv'),
    '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
    '--output-dir', str(CROSS_VIDEO_OUTPUT_DIR),
    '--top-k-ratio', '0.15',
    '--high-snippets-per-chunk', '4',
    '--uniform-snippets-per-chunk', '4',
    '--max-bank-snippets-per-video', '40',
    '--max-queries-per-class-group', '100',
    '--neighbors', '20',
    '--centroid-neighbors', '200',
    '--num-clusters', '64',
    '--train-batch-size', '16',
    '--similarity-batch-size', '128',
    '--seed', '42',
]

print('Cross-video feature root:', CROSS_VIDEO_FEATURE_ROOT)
print('Checkpoint:', CROSS_VIDEO_MODEL)
if CROSS_VIDEO_MODEL == PROJECT_ROOT / 'model_ucf.pth':
    print('Lưu ý: không tìm thấy baseline tự train; đang dùng model_ucf.pth thay thế.')
print('Output:', CROSS_VIDEO_OUTPUT_DIR)
print('Running:', ' '.join(shlex.quote(part) for part in cross_video_cmd))

log_path = CROSS_VIDEO_OUTPUT_DIR / 'analysis_console.txt'
with log_path.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        cross_video_cmd,
        cwd=str(SRC_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for output_line in process.stdout:
        print(output_line, end='')
        log_file.write(output_line)
    return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cross_video_cmd)

print('\nCác file kết quả:')
for result_path in sorted(CROSS_VIDEO_OUTPUT_DIR.iterdir()):
    print(' ', result_path.name)


## 37. Thử Nghiệm Temporal Score Correction Từ Scratch

Cell này chạy hai cấu hình độc lập: Conv1D trên feature theo chuỗi và Conv1D trên thay đổi feature trước/sau. Cả hai đều train from scratch, giữ nguyên loss VadCLIP và chỉ thêm hiệu chỉnh có giới hạn vào classifier score. Không sửa hoặc ghi đè các cell thí nghiệm trước.

In [ ]:
import csv
import shlex
import subprocess
import sys
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
if 'SRC_DIR' not in globals():
    SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
if 'FEATURE_ROOT' not in globals():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = Path(FEATURE_ROOT)
if not FEATURE_ROOT.exists() and Path('/content/UCFClipFeatures').exists():
    FEATURE_ROOT = Path('/content/UCFClipFeatures')

import torch
TEMPORAL_MODEL_BASELINE = SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'
TEMPORAL_TRAIN_RELATIVE = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_relative.csv'
TEMPORAL_TEST_LIST = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv'
TEMPORAL_GT = PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_ucf.npy'
TEMPORAL_GT_SEGMENTS = PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_segment_ucf.npy'
TEMPORAL_GT_LABELS = PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_label_ucf.npy'
TEMPORAL_RESULTS = PROJECT_ROOT / 'code' / 'results'
TEMPORAL_TRAIN_ROOT = TEMPORAL_RESULTS / 'train' / 'temporal_score'
TEMPORAL_EVAL_ROOT = TEMPORAL_RESULTS / 'evaluate' / 'temporal_score'
TEMPORAL_ANALYZE_ROOT = TEMPORAL_RESULTS / 'analyze' / 'temporal_score'
TEMPORAL_ABS_TRAIN_LIST = TEMPORAL_TRAIN_ROOT / 'inputs' / 'ucf_train_absolute.csv'

required = [SRC_DIR / 'model_temporal_score.py', SRC_DIR / 'ucf_option_temporal_score.py',
            SRC_DIR / 'ucf_train_temporal_score.py', SRC_DIR / 'ucf_evaluate_temporal_score.py',
            TEMPORAL_MODEL_BASELINE, TEMPORAL_TRAIN_RELATIVE, TEMPORAL_TEST_LIST,
            TEMPORAL_GT, TEMPORAL_GT_SEGMENTS, TEMPORAL_GT_LABELS]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Thiếu file:\n' + '\n'.join(missing))
if not FEATURE_ROOT.exists():
    raise FileNotFoundError(f'Không tìm thấy feature root: {FEATURE_ROOT}')
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU runtime trước khi train.')

TEMPORAL_ABS_TRAIN_LIST.parent.mkdir(parents=True, exist_ok=True)
with TEMPORAL_TRAIN_RELATIVE.open('r', encoding='utf-8', newline='') as source:
    reader = csv.DictReader(source)
    fieldnames = reader.fieldnames
    train_rows = []
    missing_features = []
    for row in reader:
        item = dict(row)
        feature_path = (FEATURE_ROOT / item['path']).resolve()
        if not feature_path.exists():
            missing_features.append(str(feature_path))
        item['path'] = str(feature_path)
        train_rows.append(item)
if missing_features:
    raise FileNotFoundError(f'{len(missing_features)} feature files are missing; examples: {missing_features[:10]}')
with TEMPORAL_ABS_TRAIN_LIST.open('w', encoding='utf-8', newline='') as target:
    writer = csv.DictWriter(target, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(train_rows)

print('Feature root:', FEATURE_ROOT)
print('Training rows:', len(train_rows), '| expected baseline list: 16,100 chunks')
print('Comparison baseline:', TEMPORAL_MODEL_BASELINE)
print('All variants train from scratch; the baseline checkpoint is evaluation-only.')

for variant in ['feature_sequence', 'feature_difference']:
    model_path = SRC_DIR / 'model' / f'model_ucf_temporal_score_{variant}_scratch.pth'
    epoch_dir = SRC_DIR / 'model' / 'epoch_checkpoints_temporal_score' / variant
    train_dir = TEMPORAL_TRAIN_ROOT / variant
    eval_dir = TEMPORAL_EVAL_ROOT / variant
    analyze_dir = TEMPORAL_ANALYZE_ROOT / variant
    train_cmd = [sys.executable, '-u', 'ucf_train_temporal_score.py',
        '--temporal-mode', variant, '--train-list', str(TEMPORAL_ABS_TRAIN_LIST),
        '--gt-path', str(TEMPORAL_GT), '--gt-segment-path', str(TEMPORAL_GT_SEGMENTS),
        '--gt-label-path', str(TEMPORAL_GT_LABELS), '--seed', '234', '--batch-size', '64',
        '--lr', '1e-5', '--max-epoch', '10', '--scheduler-rate', '0.1',
        '--scheduler-milestones', '4', '8', '--num-workers', '2', '--pin-memory', 'true',
        '--output-model-path', str(model_path), '--epoch-checkpoint-dir', str(epoch_dir),
        '--results-dir', str(train_dir)]
    print('\n' + '=' * 90)
    print('Temporal variant:', variant)
    print('Training command:', ' '.join(shlex.quote(part) for part in train_cmd))
    subprocess.run(train_cmd, cwd=SRC_DIR, check=True)

    eval_cmd = [sys.executable, '-u', 'ucf_evaluate_temporal_score.py',
        '--feature-root', str(FEATURE_ROOT), '--test-list', str(TEMPORAL_TEST_LIST),
        '--baseline-model-path', str(TEMPORAL_MODEL_BASELINE),
        '--temporal-model-path', str(model_path), '--temporal-mode', variant,
        '--gt-path', str(TEMPORAL_GT), '--gt-segment-path', str(TEMPORAL_GT_SEGMENTS),
        '--gt-label-path', str(TEMPORAL_GT_LABELS), '--output-dir', str(eval_dir),
        '--analysis-dir', str(analyze_dir)]
    print('Evaluating and analyzing:', variant)
    subprocess.run(eval_cmd, cwd=SRC_DIR, check=True)
    print('Train results:', train_dir)
    print('Evaluation results:', eval_dir)
    print('Temporal analysis:', analyze_dir)


## 38. Phân Tích Delta Temporal Trên Toàn Bộ Snippet

Cell này chỉ evaluate/analyze hai checkpoint temporal đã train; không chạy lại training. Kết quả gồm thống kê delta trên mọi snippet hợp lệ, chia theo snippet anomaly/normal và theo vị trí top-15% trước/sau hiệu chỉnh. Cần chạy cell thiết lập Drive/đường dẫn trước cell này.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Standalone evaluation setup; this cell never launches training.
PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/content/drive/MyDrive/Finetune VadCLIP'))
SRC_DIR = Path(globals().get('SRC_DIR', PROJECT_ROOT / 'VadCLIP' / 'src'))
FEATURE_ROOT = Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures'))
TEMPORAL_MODEL_BASELINE = Path(globals().get('TEMPORAL_MODEL_BASELINE', SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'))
TEMPORAL_TEST_LIST = Path(globals().get('TEMPORAL_TEST_LIST', PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv'))
TEMPORAL_GT = Path(globals().get('TEMPORAL_GT', PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_ucf.npy'))
TEMPORAL_GT_SEGMENTS = Path(globals().get('TEMPORAL_GT_SEGMENTS', PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_segment_ucf.npy'))
TEMPORAL_GT_LABELS = Path(globals().get('TEMPORAL_GT_LABELS', PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_label_ucf.npy'))
TEMPORAL_EVAL_ROOT = Path(globals().get('TEMPORAL_EVAL_ROOT', PROJECT_ROOT / 'code' / 'results' / 'evaluate' / 'temporal_score'))
TEMPORAL_ANALYZE_ROOT = Path(globals().get('TEMPORAL_ANALYZE_ROOT', PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'temporal_score'))
if not FEATURE_ROOT.exists():
    raise FileNotFoundError(f'Feature root not found: {FEATURE_ROOT}. Mount Drive and extract/copy features first.')
for variant in ['feature_sequence', 'feature_difference']:
    temporal_checkpoint = SRC_DIR / 'model' / f'model_ucf_temporal_score_{variant}_scratch.pth'
    eval_dir = TEMPORAL_EVAL_ROOT / variant
    analyze_dir = TEMPORAL_ANALYZE_ROOT / variant
    required_paths = [SRC_DIR / 'ucf_evaluate_temporal_score.py', TEMPORAL_MODEL_BASELINE,
                     temporal_checkpoint, TEMPORAL_TEST_LIST, TEMPORAL_GT,
                     TEMPORAL_GT_SEGMENTS, TEMPORAL_GT_LABELS]
    missing_paths = [str(path) for path in required_paths if not Path(path).exists()]
    if missing_paths:
        raise FileNotFoundError('Missing evaluation inputs:\n' + '\n'.join(missing_paths))

    eval_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_temporal_score.py'),
        '--feature-root', str(FEATURE_ROOT), '--test-list', str(TEMPORAL_TEST_LIST),
        '--baseline-model-path', str(TEMPORAL_MODEL_BASELINE),
        '--temporal-model-path', str(temporal_checkpoint), '--temporal-mode', variant,
        '--gt-path', str(TEMPORAL_GT), '--gt-segment-path', str(TEMPORAL_GT_SEGMENTS),
        '--gt-label-path', str(TEMPORAL_GT_LABELS), '--output-dir', str(eval_dir),
        '--analysis-dir', str(analyze_dir)]
    print('Re-analyzing existing checkpoint:', variant, flush=True)
    subprocess.run(eval_cmd, cwd=SRC_DIR, check=True)
    print('Updated full-snippet analysis:', analyze_dir / 'all_snippets_temporal_delta.csv')


## 39. So Sánh Delta Bật Và Tắt Trên Cùng Checkpoint

Chỉ chạy evaluate hai lần trên cùng trọng số temporal: một lần cộng delta vào `logits1`, một lần bỏ delta. Không dùng baseline và không train lại.

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/content/drive/MyDrive/Finetune VadCLIP'))
SRC_DIR = Path(globals().get('SRC_DIR', PROJECT_ROOT / 'VadCLIP' / 'src'))
FEATURE_ROOT = Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures'))
TEMPORAL_TEST_LIST = Path(globals().get('TEMPORAL_TEST_LIST', PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv'))
TEMPORAL_GT = Path(globals().get('TEMPORAL_GT', PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_ucf.npy'))
TEMPORAL_GT_SEGMENTS = Path(globals().get('TEMPORAL_GT_SEGMENTS', PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_segment_ucf.npy'))
TEMPORAL_GT_LABELS = Path(globals().get('TEMPORAL_GT_LABELS', PROJECT_ROOT / 'VadCLIP' / 'list' / 'gt_label_ucf.npy'))
DELTA_ABLATION_ROOT = PROJECT_ROOT / 'code' / 'results' / 'evaluate' / 'temporal_delta_ablation'
if not FEATURE_ROOT.exists():
    raise FileNotFoundError(f'Feature root not found: {FEATURE_ROOT}')

for variant in ['feature_sequence', 'feature_difference']:
    temporal_checkpoint = SRC_DIR / 'model' / f'model_ucf_temporal_score_{variant}_scratch.pth'
    output_dir = DELTA_ABLATION_ROOT / variant
    required_paths = [SRC_DIR / 'ucf_compare_temporal_delta.py', temporal_checkpoint, TEMPORAL_TEST_LIST,
                     TEMPORAL_GT, TEMPORAL_GT_SEGMENTS, TEMPORAL_GT_LABELS]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError('Missing inputs:\n' + '\n'.join(missing_paths))
    compare_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_compare_temporal_delta.py'),
        '--temporal-mode', variant, '--temporal-model-path', str(temporal_checkpoint),
        '--feature-root', str(FEATURE_ROOT), '--test-list', str(TEMPORAL_TEST_LIST),
        '--gt-path', str(TEMPORAL_GT), '--gt-segment-path', str(TEMPORAL_GT_SEGMENTS),
        '--gt-label-path', str(TEMPORAL_GT_LABELS), '--output-dir', str(output_dir)]
    print('Comparing delta ON/OFF for the same checkpoint:', variant, flush=True)
    subprocess.run(compare_cmd, cwd=SRC_DIR, check=True)


## 40. GRU Ngữ Cảnh Trong Từng Crop View

Lưu ý: hậu tố `__0` đến `__9` là 10 crop/flip không gian của cùng video, không phải 10 đoạn thời gian. Nhánh GRU vì vậy chạy trên thứ tự snippet bên trong từng crop view, không nối các crop với nhau. Cell dùng danh sách train đầy đủ (16.100 feature rows), huấn luyện from scratch, sau đó đánh giá với baseline tự train. Kết quả được tách vào `code/results/train`, `code/results/evaluate` và `code/results/analyze`.

In [ ]:
import csv
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', '/content/drive/MyDrive/Finetune VadCLIP'))
SRC_DIR = Path(globals().get('SRC_DIR', PROJECT_ROOT / 'VadCLIP' / 'src'))
FEATURE_ROOT = Path(globals().get('FEATURE_ROOT', '/content/UCFClipFeatures'))
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
TRAIN_LIST = LIST_DIR / 'ucf_CLIP_rgb_relative.csv'  # Full training set, not description-filtered.
TEST_LIST = LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'
BASELINE_MODEL = SRC_DIR / 'model' / 'model_ucf_paper_scratch.pth'
MODEL_PATH = SRC_DIR / 'model' / 'model_ucf_video_context_gru_scratch.pth'
CHECKPOINT_DIR = SRC_DIR / 'model' / 'epoch_checkpoints_video_context_gru'
TRAIN_RESULTS = PROJECT_ROOT / 'code' / 'results' / 'train' / 'video_context_gru'
EVAL_RESULTS = PROJECT_ROOT / 'code' / 'results' / 'evaluate' / 'video_context_gru'
ANALYZE_RESULTS = PROJECT_ROOT / 'code' / 'results' / 'analyze' / 'video_context_gru'
GT_PATH = LIST_DIR / 'gt_ucf.npy'
GT_SEGMENT_PATH = LIST_DIR / 'gt_segment_ucf.npy'
GT_LABEL_PATH = LIST_DIR / 'gt_label_ucf.npy'

required = [FEATURE_ROOT, TRAIN_LIST, TEST_LIST, BASELINE_MODEL, GT_PATH, GT_SEGMENT_PATH, GT_LABEL_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing inputs:\n' + '\n'.join(missing))
with TRAIN_LIST.open('r', encoding='utf-8', newline='') as handle:
    train_rows = list(csv.DictReader(handle))
print(f'Train features: {len(train_rows):,} rows, {len(set(row["video_id"] for row in train_rows)):,} videos')
if len(train_rows) != 16100:
    raise ValueError(f'Expected all 16,100 train feature rows, found {len(train_rows)}')

train_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_video_context.py'),
    '--feature-root', str(FEATURE_ROOT), '--train-list', str(TRAIN_LIST),
    '--output-model-path', str(MODEL_PATH), '--epoch-checkpoint-dir', str(CHECKPOINT_DIR),
    '--results-dir', str(TRAIN_RESULTS), '--seed', '234', '--max-epoch', '10',
    '--lr', '1e-5', '--scheduler-milestones', '4', '8', '--videos-per-class-batch', '2',
    '--context-bin-size', '32', '--context-hidden-width', '128', '--max-context-delta', '0.1',
    '--lambda-video', '0.1', '--video-loss-warmup-epochs', '2', '--num-workers', '2']
print('Training GRU context model from scratch:', ' '.join(train_cmd), flush=True)
subprocess.run(train_cmd, cwd=SRC_DIR, check=True)

eval_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_video_context.py'),
    '--feature-root', str(FEATURE_ROOT), '--test-list', str(TEST_LIST),
    '--baseline-model-path', str(BASELINE_MODEL), '--context-model-path', str(MODEL_PATH),
    '--gt-path', str(GT_PATH), '--gt-segment-path', str(GT_SEGMENT_PATH),
    '--gt-label-path', str(GT_LABEL_PATH), '--output-dir', str(EVAL_RESULTS),
    '--analysis-dir', str(ANALYZE_RESULTS), '--context-bin-size', '32',
    '--context-hidden-width', '128', '--max-context-delta', '0.1']
print('Evaluating against self-trained VadCLIP baseline:', ' '.join(eval_cmd), flush=True)
subprocess.run(eval_cmd, cwd=SRC_DIR, check=True)
